# Deney 05 — Viral Potansiyel / İzlenme Tahmini (gerçek geçmiş veriyle)

**Amaç:** Notebook 04'ün ürettiği gerçek `youtube_data/normalized_videos.csv` geçmiş veri setiyle istatistiksel baseline'lar kurmak, opsiyonel olarak LLM tahminini bu baseline'larla karşılaştırmak. Yeterli veri yoksa (`MIN_HISTORY_ROWS` altında) notebook **kendinden emin bir tahmin üretmez** — `status='insufficient_data'` ile açıkça durur.

**Çalıştırma sırası:** Notebook 04 (gerçek geçmiş veri toplama) → **bu notebook**.

API anahtarları (Colab 🔑 Secrets, opsiyonel): `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GOOGLE_API_KEY`. LLM tahmini sadece yeterli gerçek veri VE en az bir anahtar varsa çalışır; asla birkaç elle girilmiş satırdan güven skoru uydurmaz.


## Google model router (bu notebook'taki tum Gemini cagrilari icin tek merkez)

Bu notebook artik `genai.Client(...)` kurmuyor ve sabit `model='gemini-...'` yazmiyor.
Her Google cagrisi `local_pipeline/google_ai.py` uzerinden gider:

- **Colab Secret adlari:** `GeminiKey` (eski ad `GOOGLE_API_KEY` hala calisir),
  `YoutubeKEY` (eski ad `YOUTUBE_DATA_API_KEY`). Deger hicbir zaman yazdirilmaz;
  yalnizca `configured (GeminiKey)` / `missing` durumu gosterilir.
- **Tek anahtar, cok model:** ayni `GeminiKey` projenin yetkili oldugu tum modellere erisir.
- **Kota PROJE bazlidir, anahtar bazli DEGILDIR** (ai.google.dev/gemini-api/docs/rate-limits);
  ikinci bir anahtar limit artirmaz. Hesap/proje dondurerek kota asmak yasaktir
  (`ALLOW_ACCOUNT_QUOTA_ROTATION = False`).
- **Rota:** ilk basarili cagri rotayi bitirir; hicbir zaman tum katalog denenmez.
- **Ucretsiz katman veri politikasi:** gonderilen icerik Google tarafindan urun
  gelistirmede kullanilabilir ve insan gozden gecirenler okuyabilir. Ham telifli
  video/ses varsayilan olarak YUKLENMEZ; yalnizca yerel olarak turetilmis
  transcript/skor ozetleri gonderilir.
- **Kota defteri** (`results/google_quota_usage.json`) YEREL TAHMINDIR; sunucunun
  yetkili kotasi degildir. Gunluk sayaclar Pasifik gece yarisinda sifirlanir.

Bu notebook'ta kullanilan gorev: **viral sonuc yorumlama** (KRITIK OLMAYAN) -> Rota B (`gemini-3.5-flash-lite` -> `gemini-3.1-flash-lite` -> `gemma-4-26b-a4b-it`). Istatistiksel baseline'in YERINE GECMEZ, onunla karsilastirilir.


In [ ]:
!pip -q install openai anthropic google-genai pandas numpy > /dev/null
print('Kurulum tamam.')

In [ ]:
import os, sys, time, json, zipfile
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd

if 'google.colab' not in sys.modules:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        venv_site_packages = candidate / '.venv' / 'Lib' / 'site-packages'
        if venv_site_packages.is_dir():
            import site
            site.addsitedir(str(venv_site_packages))
            break

IS_COLAB = 'google.colab' in sys.modules or os.path.isdir('/content')
PROJECT_ROOT = Path.cwd()
RESULTS_DIR = Path('/content/results') if IS_COLAB else PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def get_secret(name):
    try:
        from google.colab import userdata
        val = userdata.get(name)
        if val:
            return val
    except Exception:
        pass
    return os.environ.get(name)

OPENAI_API_KEY = get_secret('OPENAI_API_KEY')
ANTHROPIC_API_KEY = get_secret('ANTHROPIC_API_KEY')
GOOGLE_API_KEY = get_secret('GeminiKey') or get_secret('GOOGLE_API_KEY')  # Colab Secret adi: GeminiKey (eski ad: GOOGLE_API_KEY)
print('OpenAI:', 'var' if OPENAI_API_KEY else 'yok')
print('Anthropic:', 'var' if ANTHROPIC_API_KEY else 'yok')
print('Google:', 'var' if GOOGLE_API_KEY else 'yok')

STATUS_VALUES = {
    'experimental', 'development_placeholder', 'review_required',
    'approved', 'insufficient_data', 'failed', 'production_ready',
}

results = []
def log_result(task, candidate, quality_score, cost_usd, latency_sec, notes='', status='experimental'):
    assert status in STATUS_VALUES, f'Bilinmeyen status: {status}'
    results.append({'task': task, 'candidate': candidate, 'quality_score_1_5': quality_score,
                     'cost_usd_estimate': round(cost_usd, 5), 'latency_sec': round(latency_sec, 2),
                     'notes': notes, 'status': status})
    print(f"[{task}] {candidate} -> latency={latency_sec:.2f}s cost=${cost_usd:.5f} status={status}")
USE_GOOGLE_DRIVE = True
DRIVE_PROJECT_ROOT = "/content/drive/MyDrive/AVRUPA YAKASI/YoutubeOtomationv0"


## Kalıcı depolama ve `local_pipeline.youtube_data` modülü

Notebook 04 ile aynı modül (`local_pipeline/youtube_data.py`) burada da bootstrap hücresiyle yüklenir; kopya kod yoktur.

In [ ]:
# @title Yerel pipeline yukleyici (otomatik - acmaniz gerekmez)
import sys, os, tempfile, subprocess, base64, hashlib, zlib
from pathlib import Path

PROJECT_REPO_URL = ''  # opsiyonel: gercek bir git repo URL'si verirsen modul(ler) oradan klonlanir

_LOCAL_PIPELINE_MODULE_NAMES = ['youtube_data', 'viral_prediction', 'google_ai']

# Her modul icin: (base64 kaynak, SHA-256 hash). Hash, gomulu payload'in
# bozulmadigini/kesilmedigini dogrulamak icin kullanilir -- eslesmezse acik
# bir hata verilir, asla sessizce bozuk kod calistirilmaz.
_LOCAL_PIPELINE_PAYLOADS = {
    'youtube_data': {
        'sha256': '048809136491091e9650d1783a6708cf1bc59c4a719983c19ccb1fb85b4a9f69',
        'encoding': 'zlib+b85',
        'data': (
            'c-q~4*>>Ybvfw+vB8T_(6;P%qmF@EMuuS`^CH1w&wRK75ZnvyPfkaY53k29$q++S{5p&MG&o9m=Of0zqpr~@YUdHFRL;$%)M#h#A'
            'k;CEet!%1zmdTMQWt{95dA1fm7tL8Slj10@V{!22MBIEO=0%ptd7T!y$coE!J{}BCui{cBVzw6hUq^K*^CT)(tD>x%JgwJJH809E'
            'zZ|Elbv_%3{b$ilTE<zlf=Vg1q(8-ZGT7gb)<si8w^7{G#WKbM(K4=TS#mjXlg1(zv!bZ0x{Oz1F0)LexsdVvO5{Z?;nTGUDk%r='
            'vto{2r7M}Gx!luDo61p@r>m8$$E)>wu_#JWUr8Z9#Pizepq3x%L46h1BCUj6&Sa9neAB!xpm<S9(`glost_>@r@kts6!Vxy2IXGA'
            'Iv%9URVJ4*uW3&39_AD$qFCtC#VVDRppNE6lO-Z4(?u;{%vh*e=LBszZ%TkG6v<(9Fwh_=a0CFCB8zJ&Zm%Q&BS~w3I<y93UZj}}'
            'Cj%jN#Th;?i=@e5*z!h}wB8i}w372u)?!f>OK}J-%|uhlGQqV6xD?e`=CE)ua2$JGL2E)3w>S^z38r)-pVNe8S&3C#RRl`JnrpeH'
            '<@vD6(s^0~VgW$j&zdw#qIs4AK;DmOY_AF!EsXg}maqsowiRGbU5yac;9nTV3R+LAnj0_UTipRS@V<(1Fdt<k%d#jVT)6jQ&w1aJ'
            '*?aNw-RYSCypC5eB3MZnPMoh{fOEORrJ<=|UEB=Ss>or9XGOB65xjf*0_p;)NPq$&#4W5w4G_mx#Q!{f{c7*<^d|y26sM+W7BU$@'
            'y?G2k6_-VEnaOyS^7L)b6+jRnI?tL!(i|5Jpin9Q(a5T%#<2h3_9_KjM+}Goji?|(ImUH_qU9P^pp2IiHcdG0tULTNn|y+^U_`*h'
            'g9c!w@?=%OkX=-Ss_9}qq9HV^tcVj{k7kyo)fK{z*A9?Fk&W?SI2;ZLouX&~`>vExBw!N&Wq|U4v9u4W!9cycimR(Eo$1HFSFjT5'
            'H;{oV{ky32uTtt?b-I+?72vDHC*6^L9#NS@W_8RzKNh*FvVz8;UtQx3_BA*?J2-oH8vS(e;@$D7n2I}^`4Es;mMLxGY&a6b1R$3c'
            'D|`{HvUo19iY$>Oek$cnDsLlPyA(*>kYP6jq$`UXd=^7kyJoR~O@Y;Mr{T**45&k&p<vPwg^X~$*3iWLVDReo+m{C~PX2m)6#aB^'
            'bo@FxeEs6x%U3wabME3MO=J<J)J6>(PVz{rd6^QHQQN$5GKu0EpXXPwBr;RA^pmQsn}z8BHsn+xORuGUniorgfq9WMC6n&7Dt`aX'
            '{!{NAkQ&`+9bZ<|0IbtxQLfbh61j+*td6oczii@5$wg@NQG5dsLSCf`5&^<laf`6qZsIxc9l)=K+D@yex`L&R%m84QB`Kl?kDXYW'
            '0>$8BFgQ9nJ$(Js@!K8{Z-o9A8v1~%bA&7Gy(M+8U(m+15UONc&aa}P1bm@#ylgPp=&ES&l?8hh0ou`PEGY9!C`Irt)f^BD=bGEf'
            'Rw4C{n=9p9=5wc@Uh7h>WL(?#%*oATlETj$oIx4q*Bv+={&4W>)$xliGCqh{7C1b41DqBR6|Mlu$RZyNm+^<F3!$Zq^T?ZkUYifm'
            'Bj)vQ;uL5w6JX%BCDLdu4E>{lQ3%}J6=2dx@G_2JGa;#j4ZQ>&4zz`tKN1*d4)`VUN`=<q>HhvMSCrLW60gOqnO}nx0%RJv26W48'
            '*-+w2H;pYrdnj2kr^X>(%3Oj(1+*>&=E?)q4gM(Q3Yb@Y#p5BkRRRY;Lr#!GV}HU5XCO*k%ksGq3P3!oB=G=X7bHdrU_t2;B@E;<'
            '4TwI#`H1n42Y-6^`s@G}9~djpO1;b#fn_)mPm$te@fsgbYPs}2(9(Nv9GK#p`@zxi9}nKWIEzk?58fXB5FH-8coDrgd3kacy*Yjx'
            '9Uc4(y#h=PLjD{<iPwJyynOxk==d$7V=&bF9ns}b<4-9wc@x88(?g6h(=bE~_(6i01rweNl7)A74F;SdEdfC=OE1$|nxQO~z~0Fd'
            'Tz7!Y3W$jyLW2@5t`H&=R|1NgE%p!;vK*1rRu!3q_MxVxPGe@)C=^!6P6Tg;O0`6kdoI46ekGP^-qdD5x;_ZRFe>21GY}jPVOzdF'
            'L*X275c(FxeLzegs3U5k%s?%OK#niR@T8g+w<$3V3BnDtd@PJ(Ruw?BIQ`-E+q3BC-P?n+lh?1JACBL>Jvlu)IgDN&`~_C)5ZC4R'
            '`-6cd4<|>_n}f5n<F~KiODV^|H&(#zf^zt`cZcV{-0oi7eYrpS^6UG5hgFD9PG2MMu<iHOrL7GOya~Ra1mC~;7WPy1-7m=(;ZgYg'
            '+4t%tkyLv7L#W>|`1tKh^A?y>0c(kG;mt2&{Q7-(8h-!p!vSChfNb}-`wv<>1mOTSS_&HsS(e1T2f)Pz40to*9>0mRMyfRof8#L>'
            'fGi1?4?s7XoAD3}l>v1firw#EeC31*DJqO0(Shu_1JGKemmu9GJ17zEEYqrj!SCRfd36k%k;$l?CABw3;gA-N+CeU?n;D99q2>!D'
            'L;}_37Kfh!_{XRxfQEolwNesz55eG3Jy>g5pJaFvaE%gzPg?GUrbt)@l6aZs0JVh4Gf2~@p%?)lt81NN{97+$^yBf*(VyNS`FnHl'
            '_Tc5FtrmQrhQG`L&=~$%7uPcXXFQ+Fs*3o5^}ui%od4}VeyM&L?p%BkB1t4H=~1u()|5b-iJ-*^B8m7X=yJdUr}U?TI;S-Vq7?#W'
            '7i#QQSMdtq<>)PyCY%DOE3r1zZWtwkbV2P@4Qp?q0W=M1&w?5laRnlfudRV5c?q3q&5T!4CgYAy_CsB-7&}xutecVgzfm(A3D(S%'
            '23LvqF!1TO<MH^r_lQ!&OqWenTiqK<g9vhsh*8xE==x@7AY1{2067A2NH1Y)Xh_quQxN}*fq0-A5~xDJ)Sf(f0s`8>;o0#K{2YdE'
            '+nX?ETs5;Gi<h$`7R!lP4DXiX%d%)z!PD@5+G=qm?C{iXCW}c6V?gx|;D17p9%`}#wq!KfX<Lg`i_c5l)xi;%uBwgD<dtsIncrx}'
            'xF|50@Ut)kMnIJK>XP-p&q45A15DuV-q5Vdp&kOt2<;7`#UV5b(wnh(8>ba4;!n6%ym!=QRn6m4E#j<Ni$Gh7K*B-u0?0S>>&-k!'
            'EYec#GNcUEb-IF1LC6OJ4MNWqF-h2+bGE&qRVJ;`7>LLe8XSPlL*h$&1C=;#XV}5;=hyGf-u>YiL`x8BflwjZh;?!O+yfi*cUh*{'
            'GG4}6I#yeYaI^)ivC%TLm4TWWxR^<Af}t`|lA!t({HrVsviuRYh1u0}pc^tWtxo?4P?efU&$Ad9=3*UTaiWR=P{q!k`vVZJUAb7`'
            'a8#9a&fKq9^enigh-A`?*-jP{f}%Puv<{EG08bpD0Wl$~D&TU?R^wd|xN4wy@nv41bX{3;S2`=w3q+bqr6FjBGcdJaeL=ASoJD~i'
            '=RC5dLfbg853&vb3A~i>t0oAIku#(Ep)Ci*y}W8buq%MiRH#8466_AXQy5KIV-%n(>IUt0gcKlYCNdZ{^<ww;!!R7fSb`zV5<0)T'
            'cVFOe&hPFo!t?zLe$A~To(%mlG6xjLnh|yfhUQuCB>+wXWGcSi-|w3Xi)m>NlsHWy#rTE~HS=_PGay5NDh+Z+f_VL4YhQI+gAxkH'
            'psfT$(tL5+Ks?xrgbMMCbeY!fTeO+@FT-xr?qI3iNb1tthQZR*tD-)%A}!kWigK1FK-RW(hr1TYR3vFv5sZ|829?vk0qB~xi-!RD'
            '_2y;4ZWo{$Cz~1|>dQaBn~G--VM--SAWckvxVg=UX%Mn!2M33Sc+f^3iL-PmflBhWc7SPSA?w?sypGV43eflUj3&R%i`!g}3slqB'
            'pqU<$dU<wydKMl3<;{zOSEO$`hF_(F5SC6vKY!p;lT(tPtDQPVh9Pd`I<}`$vV`e@-U))sM?e+;5U>xKq+@YfWa9t+pa1>802VnO'
            'R>3L)wBxHfE>m%hXFX6r)=&jFg>5jMLo++C^dlq6j;Wca;^1(s&0iaK9n1%n6w}+KoTFU)k;k5{YP9_%FwnFbi~k~`3_?oLBmxD5'
            '0-k`gKns=l514&<lfoGAmDOx3(R_rdZkp8C8ppSwK7zjIR1zSvDvQ5EchK5(nrCRLBGAH^1ujdPrR66L{zW<~W8h`*bEl`v9sF=i'
            '!!)CB(Fqvqe>Hj5T*FMEUR)6bozeU#0j2n3(v;1z;nn<vL6YeXgs6DEfadd9%;IYi%$hPz;_DP-hf7(4m<0uZ*GN&vSAcs-T83uK'
            'cE`hTPe2vQVTrC`jWPgg*5n}CAa?#pZ%&SOpcfe8yF*ZE1RinC;wpm@p!6nj&Ikj7Sdz&dD3%nLWh2({N04lE5s8+iBC`W%ZB|6a'
            '%UCoS!i#!`3B{LL42UE?!V5I9!4}#(dUd)7i~a$`jiPMw#sD@PixXh5CA2JbO&qpGxFp3x-D_B108uVG`$ID~1(IvKBLbz7faU?}'
            'J`5z(u(?djBxUFT8OC$#ycfkSjW(^9bfAnd62ThrcaO=0nwVWIkEA+V0Z|gRRJ=eC<6r@g#4Lq%hwAf24HW<cnrHJIzr-JLE&5>K'
            '(?{s!5Ar&b`BJU{F|f@mIuTY)#h@9dglnLqF4K=#GOa=`1d4O~cZHQ>KD;9N6ebl}i7-NmFwiISEb95@1XM7qI!g0ZQ(N^Bod=98'
            '`$m=s^$HK#LG!!aofl}2cg@gjQ6~)nxmc;}S7^(xnv9hTY;!DLk#25uc=sF|WZ9U`ofV3ZDhf-EWncy?M}6mJ4F}-`WXW=3Px<UO'
            'oFkAtnkAI5Q+CSu$0rjKP<uY7N%z}TTt#ffts-rThf<_CIK|kmqb2!nGkA&*_)Z6I?#KIw6)b|qdAs!s6r^3w+O6Pphov84*5_VR'
            '7j3b?S!PdKFF^_~6S9#3rM&?ZWGHq}(peEbLm-`H<k}6jh8DUy34(-sIUlM{lHtXqrC4l+&<3gO*TKj(b|JnHi=hbacx<Fq>`?zZ'
            ';eFWds|`bU1PZNbu*G$E6@9e5AXqav)SHbp(!?w7zV6iN#c-?*l2I1uCfX&at?&V4X-lnm`UUUo@)@;vx=HhX%s$W$|I!wqHPkJQ'
            'wqSx!=`XsY)TX;c7eQ-ej{%s(^bX?tko*j0GR6f#JCZkVL+cYLECF!^>GglIJc2A^9_K}#;!&n9!dUTH4ph=Avgpc(D3*C4N=`iM'
            '58`O|lx$3Ae_VTvW|dVChT6^pJ@DMNw(ix3zj0YtXdw%R|7lG7+sGeqJ%<HvCV`ty06c#EmOEp9J@-~Jh{X=cmpj5=NDx?d)MD<0'
            'qAmu!`Pxeg(pgMiYv?EFO91LT%{g8M@l73Mu`!EA`5aGUl{_lmYl{6It(U|uNq%@w8{j>;wUHCn?bVu1{89}J4V+}0{;(qV67&#L'
            '%!W*q*b170(!7yuujG?|J!W1&$GREM)VUx^8@E>liQj-8M&w{4`P43zUh;v*=p2DWOB<#J9f>OzrKN*D!9sf$bp(Vy`fBP_&5SL)'
            'rSjk~UtIr>uF#=m(YCnZ@$s~G%Y_{_y8;OKxINYohrNMAMj^kR4%1|$yhBIIVs`|nm1fll`5pNNpj9qSrrgt5evtE~mVu>#pvZT&'
            '>E|C2?8qOe!-ob|$@VyyKUz}17#g*QlpC$rh=A3<YpWBo;-@L?j5LOHjYd~-XHe_EmG>Kj3hddj;-Z5H%VR4xb<!I&fL?UuD*Cj2'
            'CT)%O*NzJsqaM%qF9z-YtY`<N&dNr1O0=XJZ0UU1l4U%(NEKozsHJyJ!(1%vRPPmX&<Vyj>zSr~Y;t=jCHfR6Z&X9xH`R;Nzx{;>'
            'S^2|vQ54$f47x)cNZ!D9R0HNad+&5xaEVB(GJSIEZ58XfXtYiC>99iYOjM&&AG57CbK7bn5b)<f$uB;g>DI7D@|+q#&IT>m-xpnO'
            '9Mfk!{_%KnWw@jWG?qs^pNi=*tF!tYaMdH3H0bXQ$Wx*Bk*+4}9uNVY>WN~`K6Z27Ft@-=Lvip&5Pw{$4q5AM0sMKfkmQiZBhyYx'
            'ISL8LQI)G5HO2HOWY-Ke0zqE;5*BkQ%`DLcG+tHo5g`cem+P@O&{6<9yXW+cr4%Je)WwPo1jM>4KDU}lat)~g^;flb1kErQDR80l'
            'pv)bXg&XGlVnqK}fVJvC5q(B9I-s$^!HrMXVjWKZFxq~C^2*a0$1C)L3$#q_me6v}z&kVPp^7lQHc<MhyFst0SVVC~4${$P65Kn8'
            'c2DCtNrJv%dAbqx&^KAD@p#Mp8FZhM)7QJ`8E1X|KtQP259v2oWUy(e{kV+}bzr@@mDaHUZcr;4WDRpznW751W9DOGwS#xwoPGWB'
            '=~t&9+SBo}a>E-_Lz?;!c}o|(fw?w1)UeqHb%Z9T(b91=Pz>c|Lv9IeF93J^oiilth~AO@g`P5-d=Zs~=C)=OHMct|wz(e^@NoN6'
            'G4?z`BCSXOx-_C3gAgWJu!0Q$?*PwX()(P9TpZh_*p=KZro0~81=VY>s9J+|QT5s@s^&ujeIkDU&DZ<;;tMsTC*rGb=uvn5M0|sf'
            'svWD%Zt~I(+{gi=;kHrDIG=)2`Qm4!-UidYor<lVswpd6xy17(Jo!QQS@O^|F&adQmFq0B3XCE0-GV`TCI4ToyICJ#?WN1UzkB2e'
            'EI_>~aLH@*2G#}5DN|BN^0!s;$_hixP(v-z!FUG{pPJl(#fdMiB5O8@nLU(LwjOs6xG=>c7M6YZsn^%7^a!Ya2HJOo-g4ys0>G||'
            'Wjd!2ELn}j+1ZQT_%<#<p#-X2HmiCU<FIzCcp-mt8xPs~#32sEpx}Y1D9R5V3wKQz6$XW|lOGS8bbS~r`7m-cRKlNjQDRl9Z9Kmw'
            '>Ufpbdr0B-O1VIf@KE{HRA^1WkPY}<Hn8qVk?$a{-mQ=`aU1~0YvmfLF+Sk-3eW985GVIVY7+ag;yH^;mj%Ac<d_v8J`+h~CUFBJ'
            'R4eqoLlnhc0k@uOn5$W$42AKGupA7PwVD_YnPSy&UhGo<D}-L~(20(x^A>l)e>lhv<tmnpQi@@)@*M9CvVGPV1zA+uAu{wVDaa8p'
            'CRMK+&k?Cm6k#|~6b0yLZT+mu3B`pux;uU!o6kTZ@q1uigc}Num)GzwQc{3VHANX^1bZ8GwBni`TQ#*JbvFb=z@wx65ucez{h$`n'
            'Jga4h)XdTRvVZT)5aZp{^fZbplOB)K_yHoJ#aY=&|C@N1r~hbV4%ozDlpOhhva#`r4RotcgCQ_DfD(o7I<3vi^{Or|%XoDKE1jg5'
            'c%)_|eRQ#&*TRVpVSASjlC3cC*lEq60qYe{f4^^3HqtvVS{8+L7C5kvXZyRLHn@#C%~9M3>l?roVig;lComTb|0$X(Ta)IHVmNT&'
            'pdn4$X$kQZBzpKa$M9FCr5r7(On5sLS4wQesg6b_&0_4dhA~3KDedg`YM=mf+E*~PMfw2=`Zeme$O@^ct7aMJyC^23aMT)H4x_G~'
            'D`#5mnoO~=&8naSZ45b53vb7w7Hq0RH^;CdUq1ULSPbtp4&49pVSoS5;KBWH42qub^>!sca6^uL3h4oPjFRg*`sS9y^S>SJ{x#nH'
            'xWD^9qVetp`cy?j7st*g-|Tme{aX8UPq>4N?sP}+lLv7>MsGcbBB1%;5xWH_PMvY&-oz7y9h%{UO;Ah(ulOonQ*aH+d&rn|<Q3Az'
            'fP%SA%UVdm|1Q%TnTI5yP|PV)HdFLDBhD>)ww8`$Ft$aehP#toC}Rxg;ME6d14wDLBLeHgsVxHJ(^Ibjl*$0KtIbSKTWo{Z{_t2A'
            'J?Kl=u$kkK%aU)60yeD^)22T9(N;;Z;21<k#NMFo+iB?Lxu%pC;)|(x+GpkSV9;^nWV1YwxBxpuVmm}U01SrhJJmRh%xuy1gp)bD'
            '?z$eJ!J?%Y6AnZ%?jjwlJVq-5=v!=W4RkZ0lIp^)M*6V3rwfY?W9Wg(uZJC~We%RwD)}-cov`TeR<uT%34?4J=%DOzQ=^P;Arp={'
            '@KdP^K?s<K3dMN8?WAZM|IXQsopo{NXqjivG&fDvZGgCPtFEeJ7mi}PFDy-|9!pcOBJp7#lApL|97McxkZ2+{FTmL1Ur3&`Faatb'
            '-`m}?Tt#Xmv>kD?M5HZ!pzSi|-{fX6hQfCR8Dm#z5`_1T`PObS-6HgO`^b$(J1c3HE3;2Vg5X+V!hyJ&r|DE3W0C)1J^j-5Y~@3z'
            'Yy3Av&Zq>?BIf|wH?L1m{z6pw&(KnFTYU!Qh8+5g@c(&K7B}A4XfE_znGj#4K7gh{5mako@oSYeN>~wy`!*ScqW>RdS?C~SmG2@i'
            'NS`I&t6NyYWt_;6f}QbWvz(z+hI+y==#DQB&W=y1rCD6DV3ic}1|!*#S%PAvBAGD_NL&*qrHl`?li~t<QHsQKYOBwwa;MS`lq;NN'
            '%3RrtYz8f>Dd&JYxd}0%ZfXYKc(d-%RTVJ125i7<5XXt3UZd(_1E=r%GzNn3gOyD0Y>AAppFY~sLq+Vk&UMiXO;zYM9i7;f-w$Cc'
            'KkeyJjr7yH)zj<iTYuX!0E4X&aR`Pxc8I-q1i@w*I5WP!5KkOHQcu?-@ieqxwxvVcXI(^Om@|D@zXqNhWp&W2-GZeT**?;?GiQ)@'
            '9&JNsEj(fmI$UWuYpi$g*})GlPhK6JynXIvBZ${!D$rdu-laIDDkWdziu`ba!gd=)3AmMc0NPI}!9Yfp(Q7nmswQi+|MK1?XddvD'
            'Jeb*mxu@lS(WlvOF&z0&BKStLb)*N4oJy~eTEJEncGX52LU|*wjILyHqFu?N8&}6H3{`{zGSsEM+6dWhL1Bl=1VY0$)h*sUR7odB'
            'q0p8n!!Q=8y){3=_HmSQ8Kd{|Wuzwo?JnaFUEJFrZSN1{H+M!`mQQc@2i8y*aVp<`4yQ5_jz-MCcA3g24)ZGU*r)s)rz}b=Bl1nZ'
            'Cp;K6mWju<x5<s&_@WKU=}0>=M~he)&K?NlHqyciOm+t9-v(Pxn)4GmQjzVF^ht_a)Y#DK7o_?iCfQ*(E-$GZ9pYG9NhcuDZ6-t`'
            '8acV3S#qRUQ{_rAYJ@6v7<naf$~jAZvF^A%GL#F*6qM!^GDAr|lmLWtGs>(|=coLm#kvf196NzIA5t}Yl_r|OxWj67%N-8FZMm=^'
            '8%hh<`aGIVO{PH0Q>fK9V5Z+KPqf-JQQdy~bd+9xhRyqq%n`dFpV7M6RE3{ozrylHJ#IFg^Y%yVWqZ=RY4>^-167&<y>VY}-Nw%9'
            'yR>KEStitotqWkKWfwc>(WW8UiEv>3<Z)hZ9cQF8ncChQ2B4#%2iEn-^&O)e8@>Kx{k~%xj=Z$L*Lizj7w28?s0-tJh3dXR6oVi-'
            'HCy#-@Wes0M<5Dk#zNEiMDWgh6yiMC=9VNM#!-oZY9{NrFp-CgI`O`goN`1pVcx<sn|Ao9DwBC&nhM3Y?#vvE2-Ne8FoqD31K^zl'
            'y!&W%hvo0RNQDPCL(hij$!FzJzpb0XZVa)irxqNYZQ}K$_exJtckkvgm6NFhf)N7oEiu7r`fPth@H@lar>lDOkhtb;I^XBdEE2Gh'
            '8l7X<!Lhq-sNf0OA5qsLzsfY<yk|pil?~4L<#Xusjh@7?$6jo}>rH0{+RnA-Dz7JTKCyVF_i{wX1+beuSKd}Q5mOCQ#`QJ~&ekib'
            'qL^*EiLjprahEw7;WG3(R3ihp5MWt89tk&nA$sX-h>!!N>qepj0`6IiU~`Yv)DqC?>c$L*WpT?#KkQswi7Sd@6qpIJ+EW=ID-qa}'
            '<na^oxKzi#BA(H?1|Dg!!wS_8$)tt}SlRtud-E((nI&W!#00qd#8wMi${BXEgDm={+_6n%yDzF!SYfMIyw5$h)qd0Mpy`%$vtlBt'
            'QHda^ivFf>*YmWPZgs+fFP@m?`1qT2pc7|+@EE$mhsv2>r$ctVi=AR_646#+_mf-2OElW5W4;~KUL7yoz^kFYoFu*FduO3Oo^>YU'
            'B$AvsJKJ*_)|moBx1y8!bVEffVJf02S#e6WW6+C;_5!B>M6Jh}jF(DvLkFtR=05h4l=68e&f>LGvQ9jF9QBmMM6DAK6<wUc)U9ol'
            ';dHJ*wB6-#a=W${M}2B7Xea$^2ifTR3;mh)SYL^QCP@oh%o|iYo9x8jkE#H)J)L4U)J6{d5$e3Nt$NNEZy)$+X(x(msExoMPx+f$'
            '?V;SYlgInaxf6HO-A+2}L&w)x;guOhlrZ)CHG*rOAh=o3<kE!f9VvD6C=rz0+YRUYqh$9Em*J_&pxB|kKQQM}EtVKJ{%l{I904a='
            '0Xr(l`7h3_LX0_+$RBXN7MBeGJD<yN#OVTcbQY}&>2@jy3>h=f=L_w9#;x0Nx<a=>O2Cu>rwowt>>(A=d{t*|)oxk#486U8?m_Z{'
            '$9l{zid(%Z1T`z4<M;?R5ycC#<9U851CSPdk$eOIb7YvAa+$S~l+(<_ZSo7Kc+o4c52NAu??svi6~ZwIs8$&MEAqOVX4Vtbd-1JT'
            'OrK3@9326joA$L>9jIlOxG!d_QTpw0q~<#dA1`k|N_cL0&%<(S*Y%B(Wml#QeG|^-Is5!ku5XJn+2vzDn=vi0H&U=onPgi6BL2EE'
            'iFz@CGi<=m63rV~XxWU>l$|=De;vx!rwW_eiQQlreLzP<?LZG~klY%vVdQM*7jCl05~I+DLT`||40NS%T&=PcnhU+aPyCO!6&4!L'
            '0lteOO9D?xqIOXd$C6|n;YbW0f9yOo7E!+~7ZRCMB!%%oP<Uf2YjujHT1~DZpioXIv-Y2&<<@EaU7v5I+HsAIW*kpDqYjhJzBUL='
            '6*Q?G1BQP^PqQ!f_b2=N!_b2nw>GXnrum}i`cQivs<Ou2fa**=;@qBssuEYaRM0K{>lMprndW%DL(|pxOH9_mV1Dj`yK6R6NSzh8'
            'BM?C^kpVyls`ZfU4f;X1M#a9JVx4YtPsjVF%y(j+QvvUf`4l`t?Bv!iTnszsF2pXjiGD%>)mM*LQ{_>%wN;i<KCH6lIcl{f>R6k%'
            '7I&;r&=PqDdaF8yI?D9#JUg1vqV~R^;ek&ml<m&;+?9yb{4YuRD-ziw8R;@A)wvO?Hx?z8gJHsDFi#7#&DRYK9ySGz27MF|eIL<F'
            'oi5j*gy+&bttJDWo0x~AEI<xZ9_Vjk%B-vc8d74Vn0YcYmW2uBfyIkl>N4gAO-BAWC+ot%p3&186KC94vHEk)*2E`7bbyOV*oy`*'
            '2v@{C(6pZ83YTedHQn5lhW>$!ql@7Y=Kry&mULK&8mvN|JNm1gr|9^Xf`&c>Pe-)h(3Xd~qJu)AB3df7vKRl4vI?dnGNu#z=^vDY'
            's9d@l9}pQVO-tjJrBT5b`rD`cA0UQ9ffmj?MXfz@IN^yo9}yo<2@t3_S>$Zx9&&eqOY&jbXsgGMhwx3Mlrd#(_09(AX+@PWN+mbe'
            't~qE+w8l@Z;v8aCzmFQqyI7-gVgVj^QsbW(y1;XNPP)=|`_<!*Z@6JC_C9&S^LqGz#!y&a(f9;0mi><xsVqH<8{IGBhL!7{u#~Cd'
            '1cRf=p_OmamK%}rG-R&Rc5~G&`<&idzm;U9UxdBpWzBf-9ugx`>Wfem|Ix=cPW-kFN|fX-K}FaUe9j)DwisxCleXmKlX(hBi})TW'
            '3aR6$S(k$Qy7XHRu_+Nc8yQrdWgxeC#UU*tbyrB0UgnthF`#^4xp?v7C1yuTK}KbAeu@(pw-Wd>aye9Zg}%i`JbU&#I>RR&`^){`'
            ';a<_#*@&n7kwc#pFM@{>mr%s>vZyfjfpm#ICd=sOf}u!H_kTAQFO_{+-J@akGj0k7<%B3Txv>u^A3(8)XC(z5Cq5GZXroUv{sDyE'
            'tb&IR3v?@(8QrJn?GJWC#2u2@z<P}<{3rBMp*Ty+0anz2+A0s&ltS%=fOs%@IDERK8MBuTHSZkMXgZIfvprHio+bDuV6;X#(zWX+'
            'PV8+AjE#W1Ya8F>8BY7!4uLlDQ^#GLE(}7ltw-UQg6$CKT*EEp*nMp&bME03^i8;--><o-NX=}2q{c(Rv@ioRb*sE_o-VrursU12'
            'a^kyEZ9sz_nNfxIE)B=K#PtmKu9AX=JALh`;Su4(ABax;%azM*?1wV{w22gf#(}sxln8yQ%{uQ?D7%jehbGgqx=#yj%XCKoMp|oK'
            ')^X{k`?PvnFjsADX#xk?tjLJk^2hYArDFN0?XK5%-Pc6PWM(M-U10m+21h|PVkKC$dum@8Hi3~hz<A7?k@za?)bZ}h(pCJ|J@wjm'
            'j_Hc6_apI*>!Ifm1Vz?N-ULv{m=TzBayD>I^;sENLtwuHR_n@6&OMt&#NBuH+{n|@9S561uPY68eQt|ULO$;!XV!CkUR+;fbNd=v'
            'IFRBg)%qOoEgWaX<<k{T_`D7Mi?DB&%1On6Y`a-%tGyx@5445afHxmSds=g3ofrc?)bYvmsjcsztg6j(>cjWmv2>dy$uCdjhmkHT'
            'bI{gsI)SM*>E#Iuoc;#z@+gt)Fcn{lCr?DMYwPZb;LESY6Y<|2ZsXs9*T49^ST<sk-GQ$ojMLR}y4?pLw%3NTGZ|GoXpQp=uX!BE'
            'm(~%T<SV)s4JPYb#`nk1p215GZO>^TF1&JIK*=w$x&mkC<5O&=g{YevdddyqNPBGAn`v(AYC}V@dIk?2z;qv`UVbb6W8c`0e|e&A'
            'L?hd#TD><c+RC()o9eo^73wPV<d?ly#MF>k<bHu`{hg{aj1rXs-nnku7bDS2p+=&IWuvGvbdM=Zl#DCM;p~pK#}@7U`hoq8Ar})D'
            '4bIzFGLnJ~KU_6R*VgH@;Ipel8OOp0)u_5yMLDr%+9xBO9wqWFrL0Ve4Qdf3YiThqHeRK<X#1&ojLXVJ8(9~MY#Z5ETOg;A;GHHk'
            'B6Q4E1lQwScm3E&hx1Ok{!-e-qG#wGcM&V3plot}(MD}0M9~3Auh2mAIM+>Fc#3tW6gPLFsfaT`Uup~aL`7{FJj?P`Ee0Yior0of'
            'iW6VLMJ!Xi5Klu}L-p3sFaNDJ*O5rr!L~5mvdFYwY0R9I$C;?Rp+lulc-Yt)<;9@yT<{_MPz|`bv9=&etA<K!xIt$2vw(VV8$B$E'
            'D{0@yg&rZljf3qgWhbOrk@ud3tE(P90<2+Yterb*H7Q3_=qzKa?BlxZvX0IrxQ2gVBphK8iVTRo>Zz}h){VLDz<X=mTfc&BOsIV_'
            'ectI!bEVEyS_tHcRbCcI)j&VOK7xZO`hb7PAv93-b>baic(IMmVZ~p^;?yZS-OZ^=GYBw7OSuyKX2(=r&z!6%u4hmx!jxWy1%AEZ'
            'xM?be?@osgpEFHS4jVdWVbCn+rzFGWcs#z?a8Ga8IToYcLBg*d+LzC4A@`)F&EIk1e7^q7`Jx4fKB++C&_?OtVUz0SBRnYb&2Mt<'
            'w$<p{bqhXHs8z9yUu(<hu9~G1U{s1KWhJy}wV8?1eZ_Q<uyw{LD76)aNUj+uh+iXheK1*qIf;)x=emm7-j&5OnPuvZO|{x+iM`B<'
            '8J+d%Junt0It`}^xj;I(Z1BFo*7+QH8+Ed5Y<G^*#;9%1AK}2WJP}d{be9Nv7vO15pgzI~HG=D7IeRISs(#K!Ya}ZC3op#!kB>cQ'
            'yfBw(d5!-D=Z}7a`XI8s>8xoJ&m^};r<x;6w-F7X-^#G=A3f?OcPV*5BC|RUkkm~b>CB^nFdPp<tb>U14FQt%R|OBeu>n_-&|}2F'
            'V+QW~_Vldr1;-t@fVk*A^sx$ux);$%u_}l!F>y4y<B{zQM>yG>_l6JX`n@9D|9YK4jnAFa=V<fMuN$z`A@8Hksiez%Mm_0T+RSI<'
            '9T<20>3lnc2ft227$btA{svI&WT$=50Pg5xgUybwq@uNyBQZ3VVEw{3FYn)XS`7OKW)&rL(dv3{IR#apT#|8kQ&HcP<-u)hqHT-9'
            '{ThhAIf>lBTNxB+GTKpLqaQ-&B7v>9nD}+Ib;UirrZeyM606V+<y<`NbDthOZtIw?dNzO>BulC8fpo>EF^UW*&OP0D#;k@k>R>9e'
            '>D!Sj%Y=T5{G%ffYYld*RLliGQ7+a(hkDm*@6iprvfSta>CO0}xnho%0SBGSVuzUc4|G8cmrhIfDo!pEnJZuY9ljXe8JFoQycaPr'
            '6s61ENI~CpullQVV?ok9Z{^frO<z>kMcDvF_;?B2JWJ9NmjZAb?=Ay6xRe#<?kIt)r^A-bm2Rf6c!TpM=f0M?%FY}bHhFL(N(Z%Z'
            '?n6pC0xCIOKBuN27V$O6UyC%Wkw%vo2ZQ%ZC==Np&-Xgc@Op-!9iw7y-T?iq;C77aQnk$ktDB_}Z1f<@%O=C?C}%MyNX6SOiY#gL'
            '2;6HjekCyYzDp3SGY~ZBPMEni3_3Sb3^zn?wc<ra?Jue)Q$RhtPFU#*FBFwgS&&5r)3`HZuV4)GxO6?a7E&fvgsDIxU9absI4e+d'
            'zv{t@GQm8!^(Q78^BmLo=69U>UBBUwPhZ8*6VudLt1r5B%=kuvK!b)OPl#bxfS@8Bslsp3uxDZm{DfS(SEez?!9Cyl>@Th?$Vpke'
            'SI7ko@l|a*)VO2uvXs#&Y{*Z$|Ba+m&tfxeZJ%jdb~0q$IB+@4h~50u)cXAVh!uAn+f@D<6lilwrGwNIpTbQ=wR{I<^-rfLI@7D-'
            'ygHs`aIjVQLAJ*tAnfvv45+QieKx2e=U8o7L0hplgRW&=F}+SaWMvKdRr@w2{Kh~0fG-vKVU|>>*8fUc3(g;+!j&J+ZDAv>IV}^n'
            '@TmL+TU!2?g37wLsMvBBPJ}gqj`HHv#fG`Kks!<h$ihg9pz<<&dwlx(#ZSjatxM`KwY=gFP}g3mqFZiVWu`2Tw8kOkRHk1#NCA`B'
            'R1ICN{9MH5va>4?7EIcCHWDbqRG~7~vZFMJ7!TcCW@6L)*bAF}z&T-eE}!hZwx}D)?_|O7LoK}aH|1yfL>1N_X3LPgF~`F;Epyvc'
            'Oiabd++9UNgEn;OivEpOJs{n&<J}3Jd>5`Jvmrsr;PF6eCzj~J`%i(D=M}I)je4~9)F0iq`MOHlHOAxj5w}E{mD+warn2UDJviG@'
            'n48>h+qWuNi$VJ;CAvn*W=m*?nqf)>@6=PNXfzNPOMEX$?Euj7hVG9Q>%DG=Yza51>#*0<o>Q0uRrvH7;tXHYJ<TBh*D2=1;+qA^'
            '>T^U5Qq$IYtR<spUfocJT)tIc)Z?Mh%A1LIOJuirxeDKP0W=#>dWLsf0N#Thb9nlb2(Dy|<~>|Xl|($mkeJ-2+<4DD(Os5!JJlK^'
            '4uIm{<`sH9FDdyWTAJCgewEfLbR<vWYOJg@$5;_>DIwFby;EdKmsxOG+$bw`d0w)>__t(?RsM)#y;0La+Gv(}HQkhRBy?1BGHrOR'
            'Y^^=exsu21oc`F$7m~%eF3civ;@}S)p`kZ%#7oPI&{@y+el|00hlx7DjE{#N(<nWA^_M_J^l~$xbkWZU7Y0km*7U0|{?Cl+HWIiU'
            'cc3?c;7u@6cebN<KOr~Wi(>Gzf64SvM&>Z3i$)u!rp2Ak<m@TpBc_M8EPF5BwmEK{jkS6?^U)Uz_s$3-lf2>gBpv@1$kIuj|Co&4'
            'Hk&tJAFwr(H&0~Nq#62&=!AcuiRS4->NHtw8BwV+Ju``Yr%V_+pQdBJZ~`+`nnb(ytfeUh#axZp_fg%D*z?ITExQ#E2O`S2U@|?C'
            '<YWU@Yz8R2;D_c-chtL<qJo$mWi~hE&4%I^TMmiis*-ru2phkE)1RIloV`1ZemZ#Z?)Vf1)IX$Is!k0xr0@90{r}%qE%(v2AoODi'
            '*^zp`cy;@5h^qE|aOc>AU{J5`b=!zw3Oi43t{a1i%96(qUNRjYGZ8keR~{7m@RjrEY(!+0pju%rWd~Nj-8fVS+tF@#O?x-EQAxdl'
            '*3&_&Q)--YQoZ}O8Dt4XIB1Dy8*JcMcB8m4Pv<v%;Z1ygXx&0dy6G!;;Rf8BPaUn}j#1X$q76c{dbl6-mc!$C8$M-f;5m+c{W%gV'
            'BWH21a}795P~D^;_QYY4#j{;x8oNg&X5M#S!HV(Q&yw3i8&Xzf@prU+5n=>*nK%xBO_g-%TErQMqXdVd-|JdsQPhfqK4glbx{+T='
            'dHU=rudY2Rv-fgMQ{FrH>Fv8W2jb_09}iAX_I_r?_jO$?DR%E>f7tcv(YFloefF5zh~Qt$u+wUy{c0jQGSaFva$I`jxGGTr4(>A='
            'IKJXpERj$Ez6S`2a4cSO2#-qkPNS!`=n^Wr$w&j0&U4xjrQl1tY6GQ+vAS~(_zy?$Q&SpY=mxrVg<->V1qlft#GVLi_%_AdYe0in'
            'DfU#JurOO2smh<%lv%{Uxn6$b;2&_Npi6<d?3f!+!oHeyCFm`;Gid41=EImq6|}H~uX~5pDJ6Z^(s&o;?4-PlaUXR&zbetxzZkyz'
            '`QYuz#9LrN?FyNsc;J|%BDjO<_u+G~1|d1yyOi^Eo*Li-wxPvJT_UhX^-&$J<CT=|K;48=6#GsW>~*(~bDn9oY;RQqIei1Tcev-4'
            '^qV&Va4UBL*1&^|*b^e0t))H!CDU(RnWVk#dP?JOx7e`5a{YSr?!j7}t?bnHzInOs++{(|>pr_F7>LQrH3g!2fEe{mcjV4t)Hk%y'
            'zwoF!hqf;stA3aqI#5wD^##z54aOLwbz#Sf$(#ER+ZWApCJO($bmJPSub{+u^FVub5$A6uf>Pa&p?fdy3EiVe);u~qLjNhgxJ}b2'
            'gmS_;_Ps&3t;^e@MeLaM-a>cqVebfsUO1($xak5mlZsxI8?-YJ{XZ!unkx'
        ),
    },
    'viral_prediction': {
        'sha256': '18870ba7aca2895774557ef7e2c3542fb89fd10476f857a2b858de31367800cd',
        'encoding': 'zlib+b85',
        'data': (
            'c-rM!+in|25`E`a^v;Wh2}#jj_+jal0VS3b0kRZG@*0L944RxK+2P!FdPXv><-hN#zRX>=vx{IKmWU!vPjyvw-A+~0p66YQQt@RZ'
            'm@0T7RUubAVh=Ls(KO3NC|89{6Sm|^L^2U9N;h)F)RyBNTkc2mSAixDYT}v9_Q{g*BxLh>uusdPT#A5~MH+K@AH-Z0A~$5S(daTQ'
            '#4=5HZ2pSzFqA~YqiE04^;+Z#j98kPMRsxVe)^!Ml{%|5&Uh{rRP=WuD@KWBHjPp`h?2=lFj+9E`H0V%$Ym^&0`DMenkEy5Ev(oI'
            'mgQ+!n&W1?*p6S1XqB&u62z6nSQOhdRQh<^GzxJ%CLVZH>a&3nFIXs6G8CFAB&K;G!coskzmw9Dd=P9aa+p75%e2@sXiaJtLK!Qw'
            '6hdX3MMA5sx58%PLFBB=qLf3IG-p}4jHKEMQfyyrWwOD+G*QfVw2G(VG0k^cz-;7s-e{etF$;n<9F&V7ASuul3}6v0NR13l$O}=x'
            'Y>vV`PZ<6G3p5;!?5m`VvpwUAC7D~wU@9!&zbqV$Zf;L+KimZWIKB9AcEe8C(}>~YH9g}^m={9#D9vag$RfTHWNVSrt6V%t@fhUd'
            'pQVI3y@}z#GdS>po|z{TRj$`^C22df<+QlwGBU4gPlF2&_h4}tjozNUJN<BR8@xZi41PVoxxKpn6kK0@Bqd+YtGB;iUA(>ea2ve4'
            'KK=Rj{OS^K=d+jX&6dP&epS7^zPh+Lzx*Zmcz*fz>LXsgn2$!IP^_!)3RYW~UqnHSIN^z3<Xk2}xL&|^v$r_NyPT8V9hnTrZ2E?+'
            '5eEhP7rTTj7TO$+AWt6^1UWu;)gNBX!n<cDCk>YOnuA=on9M|+6??;+KDj{DBu27dbJea{rDalh_cMEl@tBzDG_!Pu855BbTIW%Z'
            'V^j2VT)Vc*(+Iye@aYPE{|?(t*rQCs^l`ytQou^Xe)=;kJX6SwfXaqYzUPuR40z#<|5w&0qp_2=*aA7@8zYVTPBMFXVh5cgk`3SJ'
            '8W!;I_Yw#Y{$EA6p$>=xsr(0`oOVZ&ig?>OcOI=)8`d(467J8<1}En$5^|vnt3L-)WV3`PW@p1uIqH0+t8a8THmt|a&vbabGJ^>q'
            'e>?*K`D1hX`D|{Z21qUd#f>n~g>(K_{eBIU1&E;_&gSe#K-OnJmWl6ob(s!w8nYkRs|mQh8c*1iel<h2W%wVgC`9K^r#ENF^0VOm'
            '+3l}aZyQ*&6=kkFp{cdr)bQ{Y&(-NkccU#uQ<EXhjqM`U+gP7!>e8gSdmxK6cIjOEBJwm)d@TYM^Gx`rbx{*WU59!ml`IiXk09WR'
            '=Q-ckJHuk1i4(oc7*(@s6YHe`(oiJm9SrGGIi6$q*RB|~Om=X!1F6~ahP@EezqvG}%%ayOu6ijt3?Go7<MSmv-jsGnobC}vRvYs?'
            'MpSFj?(qj8^W2p%+ue!4c|iqe9Bl9BZ{fEz-w#wVOA)Mdu6vDzwSJ8=-Ckl+9qub1B!&~Nv$_jj-3~<iotjQ#Iy8PaStJX^YR#F?'
            'Bi#rwrJGeW_N<9XV-k^?`gUw(o8Unc?98JWwVQj^%PRycmLDW%?@!N;O~Op^mwJEt+u2O>xz@f@?CHqW(4z@Ea;G#;XuNZ|Qb!B+'
            'WP0!*ftm3%o6Qb~gJER)V(?JVX9_plejPxM-a1Jcl$9Z?+i*pbfujBz^F}NSIz9Egnp-7vM(r}KC^8*D<TDIT6$*CoBat+sfP%+='
            'fe7ome~jYPC7rStCjQ;A!M1)slNe&|7L`ncT(wmOh%4H~{m8<OJ^>ZCgDJJBvkm)A+`~G7lY7paVGxN`Gwp1)*Jb1GeuAIsJ@F4F'
            '43SCvYGsL2PVw4g<`|MSof;L~Y9?(YAGda*IOxsrU|r9epH-R^GATt*o|<7=5J{+uLYt{Qm?O|CG-0!*wzWy~Ys>D*jd_iut~A-Q'
            'M8E1XN+D6LxQaO**Fr6tb@}OD;XmW3u>#z_fgTMFbVS`Xg6O`&PWSO}V&GN|n}t69o*>XMcvafh@7#)}X!Fo|1lg*nsP)Q(z0_TY'
            'Q5?=EFHiu~9aonTr}^1>aG|r%9IB=}G%@7Kb1?|&Ivl+^nz3F#=!0paVA=lVFnL+ZDC}L41UxS!bs}Gb5Br8j{D3ZJN|bt4<efLD'
            '+lsJ--h2;yP;^F-FlrdK6N4$1>$8~B7>U1A(w~HYe;fn9lWi_k8a)8=T+PuDbD|@vX*Rm`fa&5)zv=_*Pvmr^?|+7`eW*Z?P4qSZ'
            'WH>ESw0_}&Z8Vami3!K_ajI27&s2+N&TDy@7$N<8sELN=Os;mCLUZd}VHrrI?NR!O;}AN}9DrIpK5~~TM+`l)hQ#jO$&rRxv}h?<'
            '9ChwyYT#@{m$53m)j`A2U~mR0HHN)qo%8@0zj3Y8zxA<=DE<uUS77J&)p>K$%Yv~s;*dro#~1xPPc~FOpFF(Hsl1qQ@!(GTaB!_`'
            'Un&V`kN+AqYgC8yGEVGuW3_zO-gDocy=pM3HN3Ox-M5;C9v|vy#xto>z<3OB84t3eqkW4JMz}U3JIA8OGIxu*b?92AQS8`XPMDYO'
            'y!tea%?_T|-bSoqU;v_4vVFNjObNo2<Gnz?q2w1dhy(TVP+zl9Aax?-n<Dd-*s+#mD~<V2JtrU7-_f#mV(Y*Z*ydx;8m5}?EPl`>'
            'NUB5<m}H%}mKFBZPM~^}tYxT2;fO9vb&EH#(Pv{Y!7s1hH8z39dB$2|DB9c2pKPV!$xzzWy@g?6Y)&1E*We%XM6OYIOi47PCr3{`'
            '4Gy}*6loUmn3JLC!e4%=JRC|R5cd0qH~(Ap{T7T22EucWW<tr0Opx3>LT?!I;Ycw-ftJndh#CrFy7xcQR*0mLu8o9_jEC~*P_wWX'
            'JLryTTQt72IQi}ow?EWQdRSbSj8@Fm^6r{_R6RI!GoZ7_by)Y!{#zqR#cYg3>vaY#I}WC^Lr*0N$#&(c+`%^;&dxKhO@D_nle0~k'
            'mr+qhB}*f|l#$H6)^N~0P-jMHrnj9Y`$b?U_9qDYCE~?QyIqny-GATde!DXs83|zB+tZSL-`<2Q5P3CRG(AxD#P?sls_Z12D_9yL'
            '(P=cMg3rE{HnqCiV@%k5Y<|40+-$2Imios&^`Sd{bt|KuJj&P`M&UQ~4bR++5589NS-Mx2%%i)GfJg^V-7YcRZ>(B7`ZZFzuVQ!b'
            'Tiv$#Pc6S4##P@}Vs=AK5iAraNeJCg=IB7>YcvyzzCKO(Ub3Yibd>uvuN7$5gl7+6d#tbO?bSdS$$cmpRqL%>Em8ccGdPV4zD2-G'
            'R-)6?c<uH21DBwcQ14S(-j++jZnkM&pe2enX)cSHE2^hENv8)e>>{0>bg}WEWDLPrKzU@lG$|SS+LEn#4l1Gpl%R7BMFiXNt&A8{'
            'iDD6x^{tu4-&Sl{<{TNiD<f22i62(y$8mj0Pha%zgre`1p^-ZcAu&7s?VQzahx!sOeBD5S>hPxtkM;#!N&!pKF9?vIZ?BKZj0tIX'
            'yn&(fx=P7Cj&}O#@|@kAeZqY7%lXCI^J{mq*3etaGOk75^@3s2C9?ka7VUUw&=I7fv<5(G8vA=$)h4w)CJ5_?21OTLgJO%JjiQQ1'
            '@6OA{zJdO-p-W@+xjoe}pTXhwt%KKVVz9G6cMsB6!)1F*wcOY>Jg-u{9DLEVqb?yNq19#91W^<RH2R^A=7G)-H`%n9*h`Nm&t<r`'
            '7-(gQo129i`#T6x$6M9pHVxFS>DoB$rKa|Ew$go1{nVh@U{atjmGr=}sQyg@MyW^N{j-KyS*%)9jM;w}{ToA|miq'
        ),
    },
    'google_ai': {
        'sha256': '218f70e619fcc6e8f197aaf8138b7c74e17e7510b1ffa1b6f0bb84d3a479d8e5',
        'encoding': 'zlib+b85',
        'data': (
            'c-ri}dw<(Fb};<EKLx6Jo?4udWIM@ZXPwo~tJq4S#+R%uPj=Sh(v(EmtSC}TQg)QA``O=f0YHEkS;-`Qw*7T}?M5a65I8tEH=IkO'
            '(Kzs;I10vI*b<qS&f4C+m-ym!9N&b#=nX{kwLcG{K<o(kG4~vi#LLW2#57Jsb{nK(k;MPukF(C!*84ZVis75Vxi~&Oyg2HM{$B^@'
            '!}G0n<$qf*eHg;$G3N1P37_-t{A4AfIP*twJQJg35N0CCQW4)r;`+vqyr6Rs27Z(^JDrYmExc&5b?YUQaXj%S;u?nW!|wJC&A9!v'
            '^SnI`z4UhbS_DxTM1DsMZ@o;+y@?OQieSEo(=-^tm$O(jVQsT0zTdjP4aT>E#!6v&zL+Myua+ssUL?G6w)DbqB@*8o-?l_^1YLc9'
            'p<B1Jz)!Z$-W~4@;s4Lx9l}~BH+}{$<GC35L39I8k_aX}S_v9)986$cUMQws)CLf@aB-qJcHlt}!UBZ1wMhI)FwQ)fF}z+*0{SWL'
            'y%nx3hhxC=K0WD+Md*zY$X>R;wI#$}M~q?8nV$+zj0u#YnfmZ7YX{N9U-%JDqb1bO@d}>f!u^Q@{hxGjFL-!@5P;w&NHf^BAH+D0'
            'roqh;CYb|4{Cyc`9t=12mSL8{5KlW|5~Sl8_BsV<F1%3?2GBGWcY!Cac~Tt)e<y^oo4nBg0>C?%Yet`+i~AtEh4J7&0HFV25e8$}'
            'RM?mJ9=pM-XC1K!#xq>&#D~s^7HJlY8HdFJriVBWEr0CL24YKinnEjrLjQa?IPMMmhjJOA54)H^`{x~4c$VO-cf`oUZ^Y9n{Dx;V'
            '$~Z{IOPJ0Gpgp55f9QxiFAOFGe1O;zkO)BqQ&}uCk*;9(#~GePK&;-$p-BC?7s&_E|EG=!<C|0-5<tg0z?ejMS?13d8A2zfZSj&U'
            'n8HZV<&PbadJzEkqp$k;NdgCV#AgrB03Kk3EKDTPYoBb1z1<E1HBJdtQh(tkF!PoMG)yw`=Fn{%16hFWL1;>{p?7t)wRIv-Ga%<O'
            '&G2vo<l1obLVu3f=S3?39nX9ONP(E_ujp(hnMoah!2Hrb8@w98ZuDOF1}Eo3ar)}j;9$@@60c8BUjxMfdUbeuaB<u}8TN*Q(-XvP'
            'U!3)OhsS-<c<ZeqZzd-3O|lHVxz})NXw#Sfx78NMbSwskDMH5Q6Mvg!i*$cy2k6%gfv7X_?{*}e@)p6)Bp#<bjPo!=;Sd1f^AY$i'
            '95BSoCet|}fh{2)rTQ;8fn6ewEkJ<6K>mE>PxL67gw`H9BF{dI1JhgLZozi*{M$r~MY>D?)qFhG%?;NW+~8824t$g*dky4x1qg{J'
            'FZ6)SA%%nS02_fCWUGarcD_H5P2&-gq8Ap^Sa>Po=;7dCcrMZykvh%1RVq;kDdd)*WzOUAj0q-C4mfs%bmxa};p{+<5Ds?-fe(X6'
            'Goa9J2uCKoM%wHzfao-EC37!qh<}I%5E=RxPC&4n)6d)B=C&c4a1Po8NC$UdGpMLf!sic8&(8V>guHiYho-1l&j{$vbRQ-N2Obdu'
            '$j5aH{)`vyLj9OAQ^c)@e~H?_hA*RxxAWw52*+;@=WN25iR1U)4R!zw9dSCH!qEmEbhCt>5ikxQ6EKz(0iF_LiGd1wcYqoBEPx0R'
            'o0e1_xXiB2kl-r`ksKH-#k0Q>Cw(~0;=F%yc+TV;1q$_{&IiSR=>T85C<;tm^B^ET5QLds10N1xyO=W(pte)u7(imwb|azyaRlD$'
            '0aFcAAGuuOkG(92qt0fqmpk{Q^CQa^ZJ=`gH39}f8xrw^4*|0liW;8#UIH_`0j2{x0$*pwx#5A4=Pji{Xu*u*@{4d_aGE^0KWQgE'
            'w`3Ogd5a?QfHQ1AxrQU}^B2<P0?!em6f#Q}kL2|WVw`kvkdXoWQlJcKHZVT2q|E=2ZB7~V*Hedh0Dl4YXBvEf)xtoqRdIrXp8SmB'
            'bTkd$qtp9J3m&HQ6)j<K5I58jSS&}dRk!(2ID;4ZmsrGrI)Li%QD7<O8Hidjkr1F0AUK4<9MK&|Yj_Lbhd#4E3D*{JIeY*o3*?Gb'
            'gZiQn1_L&5LL^S40(UFJ(=!hy5s;5eyz|BZ972%=bHA}U@bFZ$d0GIyKC*hGDL1e_+&B=(`$z6s|1}8YXTM%c0g;a+9EO9_lUIY+'
            '7iYbfz(WuFuX-0pLs$e#3?!T`;rt}P&4C5evE3KTMHqXNw51`5KT(cgVFBgI7W|t4F(Kh2e+Z{Mi4ltF*2vH9eLqs?1n4Xhs&q$!'
            'k!Z~%28j#06TmuKjYeaO=$7kFmq>bCSBVN<6ydmtid$Rq*~m-%=g-t%w{Tv=V5I*1k2H?d?>JSz64<JEuK)7YFGh6r7j6$v1Su^3'
            'rKT|<DM>wTQA;@Pna4jr#*yr}@G_i@>h})*+}iq1d`)5GJMjjTBfw^m*j6G1Iyi=%qSKZ7guXDdU_>+(NHP=aHnBYvB>>rBI|!R$'
            'sOd7IgAV;}NEm=OX-rrxreUyHAiBZo5V<^%<yo;x<7EPTWPIyIH>`({Nbw!&4S@YA4w!<V?2}>z#J=;DVX*bxzBnVqK%L9RxZc6R'
            '>BY(LeCOTS>A&?+egKS<XR@Tiz_REE4oK4d{0@MCoI?zV9D@SDO4~Iv8aIkviPJ9ibQ6i7C|tqeTcX?oL<R;-mXQdiC=4S65uCui'
            'vW_@FxZ%v{1fL_N1F4_D9D?En7=`pm(eoy7YZxr_P(kPI#=fv9P=JEz6iz&lHCXH~gTwxrgOi?uTFly23Y?U8N2&P80y<m0qodRJ'
            't^}3)my6S3&pkVp@^x3dLPknZ&Pi{&9L)pT+f+$+uvXG_CFEgBfcK!okZceT>6|R_%#)zzt#Cl-2HF#KyIq2g0e{^E{yh@%$H^k9'
            'dzu#FXqsD?!s9)VH6PW`gmgK21?V0C%0j;p?AE-q_3EtOcZY-inS0oS#RCHw9Q0x7P0~IaQfNV1zeF0B1|P-D%V5I$z!QNhCqW1}'
            '2V*9|42BgzwlaFOLgg&<wBFR<0W(mwU|&E2gQm+Ugnr`2N7*7s=00%UF>tNfG7O=6NQzd1!vbzVkDK@-sO@T~8y`^B$039<YCKy;'
            'vS0I201n~HQ#IbLHwyzGq;mmOZRM?k;*veN<hj2JqE2JW;RER1i{?09tP*5${=`vq0d?4T$0PoV5)BE3%v=~>q)7P!P(Gw>3}e=i'
            '5v(Z1W<aphF=*W==K<-yM~yrML4j2Bxu1DJi#;NCDe5>9pU-;l-Q)h@pyv)xU1Nip`Q><h4v2nn?*7s{y6B(7<4+6+(1PLnbF$bp'
            'TB0#Qni((fflDTr+c<<7;wwg4H}U_zgsqq~T08>C4T$~|YyvyCB+(nY<8<*Mntk}SJR5MFTs#je7~%nDcJQWu+;ah7=X?_Pwzjs$'
            'Ad{vNoqB_JMELut@;A?b48ery4`-j}&}f`_0UpfzTR<64+?KcnZpvK{BSo2Xkoj(H!H$T>37lyES)2fw(qDQ7%SmDPmJ`@KBz4?z'
            '2gD<xx`}&_?GysE%Ms~g)ct^zGa;!Hs5xjcxb{EzW1v$mqpwRu3n(jbr6|sA5n)KY1*i!c{%An7rjdD(mX4?asyY(YjE!=7LZ}zW'
            '1#riySVs?ewfJ9xfUVpJ-Gs(nH(Ga?EPWt@L4qfG>U|77wJKoXaDZ7-N&|U1^`LOlQ#|QFOubt;_dv;`2~eE`hzCfYJ5n@eITpza'
            'kS{NtdowXZPFEN$xDC;6#YASbWNrgYXyv8e3`kN3_$%>?l@~_A$FV1Rzn%=Z8PTeVUmTLS864poqG>*nqE6lhh{ji!gw9F+1mWiw'
            'FI@U8yx%8q3SqlRi~&3xlMIQNB5h(V)Jo@QVw<4k3d5k#*w%tAn{TD<_FG6U{SU~o4&VTe<O;~@#%eC`RFKa@q6r|MBFO_OJY1$^'
            'k-Ue)+#Z7>2AGUC5M+U200J>I3Xqqkakyk82mDGpTj%|Qv;NTS9SwTtCasg_3wT0fUo<7Vdkb0(kS4afxxo9}xBXumjwEg?K#1kY'
            'Ct9XE{>l%r9rXP5^kR7Nvd=8F*dNr{zw^h=PXD6cw$$qg*hb({gA)*C2EB5VF;W`P&_P2`Br2FtkAtJri^Eq(y|cbPc7wy}j-dCr'
            'Sv>QjQpbzo8(<u7`zLuv?<1-s>B9HNw=ODu*r)fm3uu1+uHQd+V^7d(jz;u2D)&H*?UcU(6kjqY7q;oJe=;2Oj$DxCUJZ^=7->jO'
            '+1OGK-ILyNABKRJiQhp?0EW^`8vlOTYyZt_f81^V@hbo6cG_2;c3aP%KK!2soWFz9quxtbqQsm2(Yrc)(1*6cSPEDLs!ikr`<n%t'
            '05#?DbI|;A10)M+Z7>6Z27F@s2l1;9B<Lpi7@D-KA$xCQquyj+Y-`-x7R@_<<q1E<k4)f=g^1fuOC0()ezI>4m&=SPY4l3)0nqHg'
            'YLF9olLRM+^wtX}!3{DBo<zqSX<J#f1yBN5QLwF=ktBF<-1`-T!DtB*5s(BBjTUh9R;fov77s<|#?OFB8bdOGVBA+|$sty;qrTD`'
            'k3m(UXDF6xTDk7H757DMfCgoL8cVROI+_SkW402ws3gV+K|C}DIPoj+fgyZA9HRJz!tvcx4WvhbIR;5N4(9;1e5nswzBvFn3Ux0p'
            'JoBeo!sszgL!pq34Zz!DV8VAu@RbO_ddM<U)(w5{22x3S8{bD#<hV}!MFP^q&NV3h^e=M)R^3ocAc>Udu)1TB=Ao9f_W06V&4L5z'
            '72@B{Pfs-a;Cs9jwX?Bakzzuj@6Gs92Q4sCcWIGbQQ$?8zza<>r_k*eny8VM18+z@(zHPyy17N#9z<TUQj84r!Ca!DAFi8*ahAF+'
            'YBY@*AU~34VihhzOaJAkF5Y-4k#q{w1b?~&U!)m08cK(*3#TA+-Dc{Cpc3Pd`)EIDiF{OgApSuzVoO-#w|K`|mdHT=$**$XuBfMn'
            'PA-Vy96C4!=z#y&FY*b&>->+3MrwA@E&~O{Z(2_syN8@8v{hGrZob2=eA|@&7Wy(gxhwn&8;mew=>C7u1r(2LrD2`lxe~NV?uN4z'
            '8j>huD?V%*A8_hMq__EC`8MT40`8!JHUX`UPg+Nu;$gU!W_o52e5r&71`gH|L~T;$QN`{kaoU)lF|{0%=sa0-DXpuKL!g1;IgMl7'
            'MEhvxZ;&}6x@qJ?XR%AR%dv|WPWd<RFKeae>U6jCF>I&a6m4g$cajO{AO<1=dYm{pRn!MfGkDOeR{P18hw0Sq=eTO)euKcE-ZM;`'
            'LKAf$_bmi0G%^nLei8qxngVu9>p%6Nd%^OFFsAZg5&@*3k_hPC6XH8K#oo=_+eaOZyZ}TSZ_V!jQbi~=r;R^Zw4P3I_mhUx17nDM'
            'A|dlo&^GYDsREE}D*t-mGcO<Jz<BtxqGv!IvxI1)T9CO!CaRj!fiyS%!le?sAy7#b+m-U1%h=?CzyRpfMCqUCpCoaL7o{B^q~Zin'
            '5hV+3r3Tu70`Y*>2?Lz{MSClFY~~-MDHydMLT^RmK)CQj=BvaJ@g8sPMN431upXc67WxP%+J;P34FK&M4F`4)4pY-nYBut~q9ogy'
            'E<+Fsvhi(`X;@L;Cf%+|%{KU}`s`l&`7@Br63}5`V_4Wg4OeTiw_XB!3h%vnOKf+x;eXt1L88)jdz<Web3Y7Bwbj@WNy%snq+FuF'
            'F1NI4Fh+AD^k3;dQKJe*nEz*CM;FO7?PIlsbjI^YL0$kVrlc3142b3+S_L;~{s5r^jUl{?C>YT`;w2YPOv*@f!>g|aSfJwDN5mP}'
            'z+1i(BXFZk#7*S&kMO1lczLx$1Cnz}M_jA*qi?_>mB?o!T&gCS=3227hDe>2Je*iHT8k<)VkLEeSQy4^WzXb&#yQ|yzF5-$q(Si+'
            'tX*O=$hLH)GS;s_6UPm9aN>GkY8vG@pm*Q9^U&N&Vns?cxrxVU=FpKtVNvLTal}s#j#Ga=rVX^ZM00-yx=#6c?2AuoM*osLya2iK'
            'ZoJIksh-6{gGV=bZCB6wQgyzfHVV>Rv;wQ<Ee9cEK*e~)1c`VzOI!In3#Mvd+WA$p0A^2)18=FZqA>(8kF0>)e;py2HcUv!6$WgC'
            'M>X*y8tBB0599zIOLX)mXtGcI8PErwFrQG|I`hQh)T${xb)tFV6pgv&Q_YgqzC{}pb(Oq=I^-z8N`mMsU7@-!37hI+AjLtusibfg'
            'jUi$t1^HHPXktlTp+#2#(xQ}A<5+5e`m;QWO<v1n^~<Ou-LAzq=y|91Q>jyQ?N1vrz6EhlDT`$u?a(E=a-9@SR>H4M{U2R+Y`%R>'
            'Mn*a?0w`oACNYpc-(dq~erv^alrN4`ajo@k_{23%5rrAi69ys?RU_0+8L1+%j8Z?_3FGlBSD4W+fZRjLhoymsW{T#BhIIG!vivw}'
            '2LW<-OuY2grj+@aw+)LVpx_==9pwED36|QHWwHdCgrnivY>wc-`H>$X^<oNXf7^L*{`WY3$>CE+O-nFi&>AN${zhzXsvi~NOi8Zn'
            'O(x_JqOB&1vC}2k2XyE{w?4dwPwt2--_mWjH`dpI<UG;&ojf`3w$7ALQaII8={e2=K8wy98U_}8fqT$g)6AbgbTlV<kWXpF59(Tt'
            'HOi4Sdsb@l2mAJxm7u$A@Z+Yompa>GM{G+i*m&L&w&4_}0LNz%INOr6ozNK{EA1Ayf{>_mEtb#F(7wySYnapfWOOo=FQ;KcP*3|4'
            '%HW;<h=ZtE?5UN}2fYLZRRfr}ez<VaEiGpfHUQPBC|bWY(P|a#sbuSBi*XR%pk$zooC1kar|~T=vp5KXUHMV=ZdSPKzd)_?Hkjbd'
            '*c3x#YqUh;dzvzbbH4hem3af5W9l()Vbv`(`tQn}sxlwdVtz}m%0-JzoUE9=>vQlD$3i+|IV{VN2Xer|@CXBqq)y|_4E+ctfotol'
            '*J#g|jE@W=nhlCY9EUQ(g#GK7I7v@`4?`uq6iu{1z=F2Cm6Fay`ocNCg7iNi36yR#$o*+JDFLus>|@vP*NzfUY*015MSxdS>?m#t'
            'D(DXhl1C~p(P7FVx=8Otes&)xGa;uDM0aFqFOh+Q1fIjURiIl%PCr~NDmC{CoPZh5Na(h0L&W?O(Zsv)1!(Vg%a6+)1Vy?6zyfW}'
            'NIIDNADvE7X_qs9jEEVR`eoeSvJFnk$L!r8rwFTf))Aj736k+?5{@1!rfBb?x>K-ivyy_<Y`F0iLl&Upz#vxch<secck*w*jL`cS'
            'pe-Pl2om|EtRi-pq|=Jhs{gDsaw*YW9d`Ps%K}Kn@opJAWS(~on+;%6W?1BTWOvwnqZ=o_`Zf>?V^F-P3n#$~d9A5#LeckX?x7bJ'
            '`ijrTftW2h9OYxD#+0<4ES3Oe|M@N0&#@Kx1uO)fT_=nA_0Dxh|C=l(9B)wNS$s}%GzAJu$HncGW9-m*P`R5fqW~ksC?IZ(ejDc?'
            '4WjkPPZx1SQArUXB{VWzw54PAJ!od2|FEAug^7J9U`pLx@;+Y7;RoUgdm4qo4OxSvTI4OWc<$*Sv9Hb_d*q7m4*P#aFSAB5wx^*Q'
            'oDa_~4u%&Xk7BbtUd-iqG2QO<+3Cf}A-9q!!ZtWp2OQ|O!y&qLG_ow!kNU?ixvv!)q+j=5gEWk<y&J@~v8&<>lyHR}4CChaEsTjo'
            'f6o@ew<R+C9{_nv*wHj;x9PNCN9RCFx()U=Y?ua4)3L5gK6r!>e%fs`;3JEmTLzhJ3jbvI5Ad8C`|8S$|28fx3~|(ji9<`Ae&y?o'
            'UT1Z$VX@TPxJYRf)~@GKcy7*zK?5!Mhw;X8Io#*_JoA}33PG%~&yX{ovu^<WuknmU2KL8<mkoI)U1M#^wS;@&J-i)IpnXH=tcc#e'
            'z*wmE-<KZ707^0S&X)oZit>*g6Y0Ea$MM4K7~OJ{C~s)DX`HH-D4ihvNF5dfR!9zhB!ulg`El3X-Q6X`?LOJf+mvuvhk~^@60D^I'
            '!CEp7OnyN-G^wTh_}_ggyu~l&7K0SHocm@tD@lo)P{K;MP&m~$1I}I*bbj3ZHaordHCFUrzX0CfgME3E(^ipXLPwUVi~vtMKN>-8'
            'L`ED@y}wWX3Gsd{Fz`RgcIu+|=mJ2-5=V13&SF?!MbuuWQHdi@z7EzGab)*ffubPz9$x&de>6Pq?;~o=gHnJa1w)cEo%r?iZ3|t8'
            'S4$%}K@X%*%r$}+GxT8g&<Kgra$t(a73g}nFHp8=F4)-~O*(k@I1Xl6APNrnj@X1AsnZLxQd^F?!3GV_TMkJ;-KV?>d&VB@8GB%f'
            'Iq2qHBipzX8qp{rrCKOK`h{F5F8QOE2g>LrKZ<{HI9%=jm{l@n0CwhqrC%n=TQD<XWFLnV3D*i%d%N;rJ~g35Kl1?)I<_SK$RX4r'
            'xu9L05BqPA`X|Tzzey@&gw%o>8Uwrp#+N1NfUAOZ3ukfeXAwH*ZsWlFqSNV!f8hUaqb+O}tU`2<_*nGUuK+w{ixJIu0q0%Da>pEN'
            'o5xEzVl%+cz4o*A)4fqU$Tp;s3O^;t_nsRdf95{J--Z%0_sY1Z^h`G$-}-aUvLC%5KTZX~fHJPYW^sgsw@N6gfniiSc(j%$&qr<V'
            '*?;@>@J_!1jRJJTo5Z_AY2Q*xS|VOxW*Kq~s^4zu5}%KvDbntb?I*iWo<9yI(o_|ko+~)*T9nB+X$o4#Hyf=Af>i_~UM{Rs;BMZ3'
            'coMOL7jPsN0h%h9Lfv=?4hv_D_}-nDq~0n3DM7sLz3UwfDxvr{KZ)CH45o7uq*mUceQ9n%&lVl8f)<uMAB!9e|Lgzt;OGL40!`(L'
            '?0*=CONzF|@LZKT$Z2VxoFzthiFUpG1{3*a(wp2axly|%sDtRB;|x{!(Jlomy$~_mYi1-gtUS>>KkA7ajFkLX!}pN8!ziD@*XsnY'
            '^^>n)e_ybkN0-7Z9Ax|(|MRHHV^9Z<v5c}M$E$*X_w{B?;Q#M5<6KnwXP@;0#f=Sb{fd-Wow`gvG6xlqhQZ0Oe|FwKz?@fXIo&5>'
            '^g?)2tY@D=vc;<WP$SBv<7w+Vg;Z(_E8gjLq{WEB9gha*L+wUyTawhpz^#|MQ5<~)qR4kzob$*Hj7!_x%^kM_2DFEwD_hmey5+&<'
            'rON@1_ca~Tq19#4K6oYm3|m2&K+3Nt{8}y{NL_79=HOIU*_zibh9X%Q)EGODS(mB|)9h@E4s--|y~n^FFnQRm%t3=lMqpf%olQGZ'
            '$d#+v;bmaK(1$Cp-<xl7*D%B^Ftp-SG~dCueeu-kh*Na(REh4UuzGaxWY<qb)pSWAQ`2QAra_Ws6b6h@pu^L5=Xs6;PBb(3kgtkG'
            'u)v#q<z6-Qq7)CtSHA%O7C7v?J@hr{4F|taCV{f8vsGa<Y}FaeS|u^rk)L_#zPvbkYk#7mYlX?GRVl7I01Xt#+Pgos_kP0b>JU_k'
            'AdIu+z9<q)nUOK4Gcp?Z_37Eg$;GdyXD}FOaxltG5{PHJ&oEGVxr!6KWN$>VK)+nXe(m|1#jrc~6h@bJ#JkW(d{{2f<p;)E#Y^;6'
            'nNT}uBNdOG5^yoA!HnO+^zkYg$teh{{0xTj=jgk+V)2(E0TK5-;iWL)RUk6>2ROTk!^twiDU*V=jFw5boI!tDl1$DJ0=b^xfE4i@'
            'cEn3<3CXQ%lqJy<D<Jmp;f~zC1M_}oLo88Se<<&Jn(M6^tw-zuYPi0>D{59L!!6bBmA*7;u9fJa!`85O{??^#M!fNDoXum(q@d-q'
            ';*pWend5I%S}=~Ft!avpilo$xSC0-W?=8qGk<~XFEyI~h?k<?I$^JecJK69qNIW2qNXP*Q*_xIrT}9QBtcH2k3z&iLMaxBXyOL>5'
            'H_I_g4N0~i19jx{+=!rGOH^o;)VA_xBsM6y<ei_in`l~S@kw*EHByi@vLkvUOw%E{e@5Q|+f<3<o^yB*-VEOiPTmeqUb{#AUobb$'
            'It2Pub(v%sJS*;J^xOLIi`wf8vZy|+!P0LtpA9Eec^1CWj5eH7@yLCH*=#nE>O=jhzO!+?Qg`*E+dQl%2w(Xt)x_j0f25KSedTXd'
            '5wU6nkrx?N*daGrsz)ePg4B(IV4kd&|HPQ_ND1JWvQ}q%ma&kM{G9dwaxpmTAG)mZttCIEQdmzE%B?C-YPnhQ^psmvqjkAm4Q7@b'
            'RwGxrT_t{$n^kRb!DM8yDdQ@lf=gNE_q9yQG9DI?L;0Gsa@?jwAY$xe5W<a2Z%V1|K$h*j@BJ!X;ygL`qlY=UWoozdma?T{Z%z`8'
            'SQcU6Xh`nk`)KczHgQUCq!^-18IDYEp`2D(9mkXdb04H@?}Ky);xr}cHS2-sdMOhNWRz7BaGVo*lKPADna7d85~yMAqXwCQ=sKpT'
            'JRF^@!Si6!<^%>UN<p3~=X%&`PH`}fQ_A;v-aj}!Iixgp{@<5A+8HS8nhau4)_^?Yq?L7$5GY<^HbII8;562p#}02QGBeYivSQ14'
            '6qO_tbKV4$7u*&mC~;Jhs)TZB7K9-=99t>F%W&+<jN_D3ER_fTxc65VWSVyuLr(p2Zn(Wr1|-odx;u55C+*9SA`p}@Vi;qG7N~JH'
            '|8wuh-CZ#}L$8TRs+?=T6BKL&uz5O@h~w}Y(<LZm$!dr4*r8IT68=;mPN$X{?LBv&{`71OA+j1&(jcovk^Go16=+_TLBmuY4@jP0'
            'lgtwfZ$;Tf$;8oFN5Fn^_x|`~J?S!=R1+<;MJ2g18mQ$Q?B}W*^q`!Cn01;L!<l3OwKyFqdnpEy&PLa%CwlT*57LNK!b}V~Z}Q!o'
            'G96*6+JD%|*3hb9*n1?=l3|bNZY)>^m&qkK|C>HeXgi8x73NMfunCmbxK0FOLZsv-){$|wIhu>srRuXp);jz;+K^N<4z`l<Dbjp6'
            '9QKdjkyiUmLdeD{8$2yhHnbJ<Sy12~PK*zc0X2K%7Fju!(Tyxfs`5dK*~U!|eY_r0LW9a44z#4nccKj5{J8QbF}odf$ic-bPghjw'
            'IvQFnjPk9wbkZ0lQ>Tt8+rVzhfyCziNmXpBez2vKdX*d=TcYX6FkVh#^7SKOa_LP~mB3OIb1w>}$UCIT@X6N?bjgwQWbmqgK6Ei!'
            '3=<U?p18Se>8crZFoRs9+<Q1Ya^IX@oFQ&MdDc-@WOo|);iL(SxJ5DUEp;Rd;Vy^4IBT=1lX?KRiD%0N9eGS~&zw9}sfmKb3@}Y)'
            'm<SFcUJcI9hsu%<EFOqfFy6P)^XsH@h|1UO(w|=MTz5c#pVN<-ze=w;3^dD<V1(vdl1Q#^kThpWQ{P8ZiUYCDgKRt3zzZ>1Op4KT'
            '8P!-Jv}0rT2q`Ag(c5^U(tfM*5t=-SJ0D%&OR*^cg9=xF9$rFKH8nz52=o-DU+M&}bh-__F^(M#!K*FrpALawQ4mBQmk%AqQzqVh'
            'DYCC<Oid)&;)H-<F!RNBjlpwUQx(dS&{b#!p8<6VpEWoM0d5R@+PVkHWJGf(1$hY$`A);RZ0}v^gdMyNoRzB$wJKVH=VQlpe7|96'
            'hn<ZF7I4rzIXxL1^p3>A=`kFf!+lO!Op<NQ%@$u#u~3;8EBcx7T@n6I+wlxT3%T$1zPJn~=@mSngcRHAhl_zZ+YcQ$CyLT|A+k@L'
            '07pw=Zh6XTtZvJg*z*kx-RJWaf4i+O%<gZc;z_b9shl#zN|w8!dpJ&buP%8+vyf`3OlNfL*8!Ze<(y=Hk;^?ikNvWR9tC=xH5cb('
            'A0sQsS?62p)<p6YciP-!Mlu-%T@z}9!R64Rbnrek;u#1_GCrWeYk&uruxSn2s|IoWCUkbxBUffwGLpf#DlgPSd2x!J(DGz7`KRjB'
            'I_HcMbDdWcBjYT<J~%Z?V7GvL1^HKvmGJqnSAowSwyFZjYk_8adeuVEZl<9r!?hL4(Q|~awWn2`)r_U|zABAiCy=U$1TACt7g3b8'
            'c1q*Fd=Di3lEZ<Hj8bjHIgMOopd5#x(nW9jfx;jzN^<CjTh_%|)tZm9>6fDTesf|v;*eQ8rJ-RS(;|$L78Dg^CB&pw*M;l;Yx;f?'
            'FT}^(&q=xkw*M^i$jle0T2TQJKWS&i#V^rH^mfu>^)f{^qmwn3cZM8pu5lz(xsj?J4^9Thu-<eIniS`S_HHn$Jrk!VN5ATb-9-|h'
            'k}U~12p3r7P1QT0vPI?1Gs1##{`a?W$b-R#0gt#*X&!T6j&J?(Oo#j8`K0P2-fBC=9L=%>3d(3SteP1{abpl0l|tdY0$IzZ(Xi=&'
            'k@&yRQdOepz~4<(fugg_##&J*p?g_K%0=t$CXD`(ki<=q*E%x!Q7*9%IC<4mHEC@1H*><bIfXC5>!I#AZa~1dO00$$hF-;`n&Fir'
            'v#R8N9$j%cubLKa>b&f{aizFLxJ07^;GMhqXfi9++7WY)kiuZ*XQXCY;Y8x_aD1}Yk5JuPT4)AS0|_{bzPLvrO%_bLqq1|1z;qVb'
            'q^ea$<7~+haS$qV1-b|Nns9<`X{pq9fNic*sRK%}j0l{1Bw?iHdyyQD&5$FX6VC2{UKL^1wRk$^oUS0U-`sEto@In##2~E8NPE37'
            'S3L(nQ7E8+F@W?}mKmVjD?cxl;4V6Atp_<{s5`=3LU`&Pbg~@NlSrj31AWQZO-K9#0oCMbIA-G3D&xeZ8u3cEGryj{*0$L^iKfbA'
            '>E>afoY9gzxzxj}q6|}t5Uq9$tM8Rn_sZ7y;&{?(FSA0gLIfxGu^twClmk-@NEBa|AfjVbxxM_eBskjfs@&FkSnN^fE(g6gai%Ly'
            'Gbdp%fQgBQjBWQ&VY56}{_Cog-BijX7NWB)jvey?Y2#$S7Pg5rE#m>lt0>!E;Ue)8$GS3e0#jxVkRWCORV(O{nsb>qDV{W%MlnYe'
            'r5-^&je)1!!Cii%f!Na1Ye~}3svsYZHFz*p<Cg@7;wm~=a3Pu$td-hf+T!H8FRxp5{lpy6*i$O9g5SpgZ2DoUTeMpY)G|?EXvk};'
            'JjpFRF5(lnWuS2WT;7-hnc>-D1O8PUMS@pSp~ij=@N&}MC1NAeb1PpJr!<aYITw~f8YMi{49#41gTp^?CsY2sT}Q?U3|A#)aSANu'
            '8Q(BYm+NH|YJT4d(pY6lH>2RQ(7gpZglnlcdsYmc<C~!v=MO7;D2dBphy3WOzPXafihcOG*)bPAF~z=-)ufP3x|G?nHVsalC`^0&'
            'NhO9hYc90%e9(<$gcYl!hADwIG<Q|lT;S$T^)6PKAk6Fo)nX&GuX|P1I~O0!j`>rgkBQudfG!0EqoHnLN&+VSVTc@)0X%K=!_MJg'
            '>T^n3H?v4o<yJ!Q1Ei=-k$EoA$mgPOHCJQcVq$1%sVX97^vjRCU0bTy!qud%5DB>QGe1JFR(b3Lv4TAe4Wr=?6sHtoUDzZDF=V-_'
            '+>Z=bnul1-0)0}kN?Gzky80G!tx#>Uk#`e9k1^H=Dn|uMYJ^Op!hVvAadO(1r42!{ja`*B$bM}@VKe=+TAQ@#ubNL+{b&g`(^TsJ'
            't?JA@s;^FWv?=qI++i&buDZ=cT}?>>l`2~~f4;^KAYG}<X1$5*O<doat@>g?psq*)sBm=^&qVW6VHlPFuU=DB!b5p6l?c8on#Hgg'
            'jw}C|f2jyHhI{z~mQ|PI{kYWDN^<E64UPHr%un&POc>x{RJ(OAE-y&=)}PVF1+f?CNqXB<Ph>`LZPx>&RNQqR^v2jzt^hK1Xd$*l'
            'f>cWeGBL6??9nBz&DotbkuvaYjN?%=po{VRk316>L%{w@E)}%aNCnO#c6n{xLg%Pl{K$!ky_hR!Yklg~(h@EESSR}!MANw5KE`v('
            '+b2F118j1?ZrS|yIi@;Q6aibejjuT3Pu;xCY?FZ6C0u9%kR5$N7kd*Igc>%jSJ~77)0SbD0v$*c1zm4o-}-#x9ghdwG)A$Tko(JQ'
            '=o?t$^HZ|vnClPI7p&K{O4gK*rfrh<##oX%KkJE4C1S7T#_T-eRd6j+=LWju5ZdBV9UUG?19akJrP+82KO<FjY3zq#?j4w59qBZa'
            'SCVAB)|bbdm9^Qb2Loccx)L{KVxykWnxZ~iIe#_t?iy(8h@ILL#kKw#I$g14GqX^XEPs@-7b!9>(pi)DdCdFRrDA~{a%F2$XLOq2'
            'Y>$34j?q=2+gN7P_D>DRsTje{;oWTVBc`2(8X_HZ*at|1(Rl<@q0eJ*HkA8jaXMYIszc?c${dWx;4=yTZ!oT|{$-l~Y*2tyi2sr-'
            '*tiX{a)lR_<B^(gm}HwgLB)4s94E<gk#T1{mWfYPvZzKsdMmm0b>+~iYpsfBIZ%xgV#w1u<nX$iK}Us;T|`jei@-X^grP0nZJijq'
            'Mi~D#*Rom`mm*L7b*WRPs>keSsxwjW^&j1=6|eu+T>Ys#Pw5W{wDt7WE780@yEsuUiU+4BCw=AFey(c>UsFz9^7&?h>vTA)wt1x|'
            '@9|zx_1hH}LLtv@bq{Ob@G;4=4LgY!<^BX2nX_EmF~W2>kVLa^<4-nE)IoI-oYWr&93mM+@Jq%_`qF*OpKyxPY&pmJ&&Vt|{WkhN'
            'cXYnMe<sHf29xr8sE1RHFk%DlOPx(DyD{MhqV&FPv?Q``it3uon--1S1C+dHt#+2h(lN^jNK18<xmJDKybm(%*}3Z}`gR771iCv{'
            '=4VO3sIpEYOfNNTuX0;2ehKqt(g(HWSKg<t+o-y5G__%kLIHI?@~Fi<IF*ybTQyx5yx`4|1<}&CZ=_I{?DAqScdxdJquGdeRc{wB'
            'Nf$Z<qFeGcD(2N{mY_lnyScqVHpE<8h?!+D^3_xKQ={aGj?U^2b_Zv}T`ixg>TNa*{A(`q^VCKWpv8QVtyFqcSHB^lOa0|IMc*oG'
            '%!m3K9VN1|5rTYuK-3YZ-{Y+HWe!^gz;q=$T;(@#_2&AwwkMSML9JC5LST3AKYV?6@BZ1@>6v@h8}{9!!SP_|p1nIpk&4b6*+S_a'
            'xT@V@sog}idpS5dLbrY@{Ep3l7-6UeD(Y-+R(HQF;K*)Rf10u~FSA?q?Bo<w8Vo@rL62QmVql<Dzelz?@1OnBKcj@wl)*(_m&mun'
            '!EqmGJf(;Z=6;M>re(k0tN!p;>gP>;3Za_<TM5<s!O1VZqrssJ>A-eDL@J_Nk+_<>7<$L?;QSnbfZZy%rEGt3^7iEPJ%(B8TMw>~'
            'jw_HJVq8%ni?Ps~48cjp%ODdY(7a}TvYk@JVt>5Ebi{4t)<NB{0{ZZ*cXB>}S-Jepw}X>I%AjCEuO%#KIp!?5q#bDC<J&yR2=W#z'
            'D<$1UA`HKw{poDI>Kz@u>>a!<evi&m6~?uyo*t^F8U<R$0F|>=f7%<@+Je=nvL{D_w|(~=M%<j8<dd>0iT4H{z02M9AGwIQ`&;|!'
            ')820DLAfVdZJR#_&Tk{E2ylrFe+FsyABetjBG2wWM!uK$$v-}{k=(SQVH?^&pWo8&FZa7w@Xv4OKh%!hy!`oJubMya)3}Sk<quF6'
            '7-m>b*SY-nfBh}}t+9Re!<s3kv$jzc-CUES3s9eHtUe{A=E4<uv8p^S;|wbaOgO;G%Xzmnj?x*SaLZP1fuRDI?1f2pwG^f$gNxEt'
            'l%bV;Oiqy89pjSD`^P;vOb38Zzk+TTW3?P&g}Ju@dHlC9nzbEI+aLzt$RHrlenFVRQXn1Ru))m@2D7OOC{!c`)MHfgD1$c!Jjq0k'
            '6okr{Ox2_naEL^lF@<2rRFr*@QEGEpN?<Arhzce!GP*7WRjH2@D^v4Qm(sdoiu^Pc`l;nt#ZZc4EGuc%&2Gv<+QefpvLU*0@Wf)G'
            'L^#42PLX2nC2oKWQfh?n<7B2&TgvL&JA75QL#b%eWdceJ$KwJiQv3_kX2sd9uCS69z?4BcMtBa{@nTHSMa4K&O)ThDig7V6or&F-'
            'H=}IOH)(wBI8_*5H<V3P>V~XQ-RcG{GmI_iH$0I=*2t_#LpShYLzXTxjgI_Q-lUq1SVu+D3=BpzxX4uQgoEX3y(D8VQ2a&JHZ3X0'
            '{)EH?HGHdb2Kig9KIZp`2r~SVVbclYdwIuzGwslyO-J9a5+rb5&vy4(;@R#~T6$xF=$q1gBS1Q!I@ysP3tKEBq{LL<0c-zR?|<7W'
            '7!I+B0NN^Y0#+JNmVI%h5MaKq*<2K8tYvh&ln)9=%cYp&ckaDZ7&Qvl&RSC3yq@*rlhW+8G_xmvq%l$dLjO9~|8VOq)2srt(Dads'
            'u{T7qABu~7#a=y8n2n1eeyNLrU#=1x`GQoob}H)xRtTPTqfS6TT5Ue2EtG6M-sZT3xtPmd{J8rB|K}<G&ojnh3NMI&QnJ9h9-@HG'
            '%00aVSG+Pqi+sga!523`3Kpl-#I>MY9E|ScMAe&#?@iFp7u^7Sf3guKN~ukMm6>E|=n+@*EIkLNc^iy^%t$M}9)iH`W$^kY5Xl*b'
            '^;}Bq%J*SCUR7m6RFR+2syX+P8Rq-TC(UKX)zhZow&vqaDrWP=pYoUV@JrnVIp(Z5OAK2})H~JM@Q%WH%2S9PmQlEz(Qpgp^P{g<'
            'A3cq=xgM$2j-}@b_N7*l;t9}}d!SmkD>DnEj}GQ^U#!Y;!;@S<k{l?7A6~@z(Y+Ut@ZbrMXVA)rQV)t|A!hrCFja`7Bu!3ZlK6ne'
            'E0hbdgbT|ZbF_k>_B-}SXruC+C>&B6Kopu-N>n1E85Pgb$K)b{2}H=39+vA%UwxUzQEErF#runJ0h2D{N9B%CJQe>$xel|Mn2(gK'
            'Y~p|)k$=1~z`_Heo3dx-?IjtP<ypmUqgY^vvB-EYwxx?UJ!<SXb{pGyC-ssI2Bjt%>p(zA@s<+c?UM3|Rcm7qoLSDOjun3sM<iv+'
            'G_GxlG82r4O;+t5DX}}ei-&&btyq1bimZ`}{yIW*$D{?#K$cv0P=J0j9KI9R#z5EN76=~J_44o{8z(x!F|kUR{y-sAoSY#r&Ky36'
            'wttGd2qMm>fQMQ2Lp$!%y|f{wiFY%n`frhr+mHvKqr-gdR2MRbZR)Yp&n;&4u~3fZof@*GEyF2gE|=mOvP-?>S!hrp8p{@?kaMc>'
            '4pJ8)c*B4%Q#bt6Q-6Z?e)em-_X;%kMuA_$x8xx5JLL=4lEJw^^yl7(=57ZQ&XbFY9On{-y>iO;&eDR~)a4ypqNFAl|D%eay52GX'
            'j~8tqk6tumfmKZMzNie~FB9JC3Itlp3lL8Blkb7OXE~ve+g#Y0{Ik%$BU{ro2@X>A&2oi8_K~NPi+5t<*hjHaon;-8Zw#ec*A8cf'
            '@N-*vFO1d&ygTS{(wSx=A^S6d_P55*hQX!rbMv>!56z$Vf9t@%&d*NzbHmV9aFl_@Vj8;a?MG-r?>eZ}FPeLfWiVE$7bG98So?m('
            'Iib`AKzlNYh)!8=EMO{*<kXFXyxDX{)j*=30Hc2ga1>wxPf`Td{{fI#TU3U{M)1fHKqtwd{0#DWk!Y;NfQoCQDwxPIoYIxYMhCW-'
            'Pp0h2xY0#~yw=AXrVwklV?v`Y$o{(}yUmd3I0yl`<z|PJcIbk(8p^(84&A1l3vAEnsH6`QEGQ|Oo+B$GU@|Qu?~kl4&&kU$Z3uD='
            'FD2D6#JB_P_0k+6&s!1iU<qrEF4rq@bb9*soKtdmNN{&7OQ)p0JNY!F$I8Xo5pXM@5xHGT=WR(;Ug|W96UsWZy2pfSFE}R=Ww+32'
            '+zUAy*sKAAR+(W)DkTTaY2y=ClA)vBaVhZCbszS{CwTBsWnrohYvBBF>#OO?X^|;IIZ74_#RB$!|NZx8{lgw+q=%mk74us^6Izh{'
            '{Y^hw_c^xuWT=B8FRC<=m%br8bkgOh8G7^41jO!r(VXj3FMcS0j%WwBNSktBijCG@cH_sZty#2Y&f#JcV0>}8FOT2VH@LZb=Z*2k'
            'u8qnH>SmnA2OVv&PSFwtH<Ui#9kNn`fRRG8Kf(Op@>cKp*T2;xj<ewDx$$%gUuRderc9c*%*OlLF;}Hq)AZ=#T&+!93=aekurJzw'
            '5R)a69C3I)Y>D<yBF5y0_d%+3wF~s?ME73uCuW)SqoABCn1Mx^FItQMB(9>xkjg$MQ~(@C=8TYB(#;o70OQXQtR&tYQysOv-TmEN'
            'nwq?P+DkK|(Mf!VNs|(>`$yT%VahR>FG62#OI~0`)}0Zg7PqDTiDqh;vTY^Dn{x3!h$iuUD)Tmd&_?X*>?1Dpn({Yc(woVe6f;zs'
            'S_4WJr~VuKJ%GP;D+W9ud%kNOFI6toP1U~H=mCKa#@^0RoVvZ}#t;3p;W+4H%|kRTCN77q6|v;`0gPyJ(@q9h8*16b5j#8L>2qLY'
            'kGP#{iFXAenY|2rA_cdEWD-QiCO&P6r!BE(iVkkdTicb5e=yv_;d$C^J{N6%*SYuo89YNDp8qTU08Rg(JkljjSYO)P)1RUwWBHeK'
            '`|6LW*M;x#P!k*|Lu-2O$=|e=(&5kB&*pK2u8Q_U`2RI@5sM;X8J0Rfe4GhdB#e2O>y*>59?4l)&(=-ikLzb3aS2EVh8$?X{K-TH'
            '@h3e|q6_Vxv>AXY7<?ZM@0G->!u#A3t>9p9%k)^3h6J^FB+2r2T#S57V2an+EzGLIlt5E&Mp0=*YAd2hG(x+N8w=?QV~*=%A9D2~'
            'KNNLw7}ghz_K(J4)O9q7NV!uOkG;@pZyf@5Zhnlr*sPd_?9jn;%y3${&*(d#2PEKby4C7c&bojpu=tdz1bXb5P~1b!mj65Kqh<t;'
            'f_?KH<Pjuw#0{Wct^$-9g*1-mqX2Nl+{Q9t+;cuF1CAaY*lUMKlk+$oshViJc4>@Vbw1(9tqF;&tf@U}!kzX(4wx?lg#-kKbHF4>'
            'C>0~7<cw3beJUAJ;-$)0A04~mXuM1kG~!{5%Gisf!V~%)CDgn1!f6{nm-&&D40m+=iw;pS0z>e39xl>ZcnKv4oWZP`{6{LwviTn{'
            '{xP1>pHg~;^>n!-3NAq5r_<mA$8qp)Sktu8fp(u_GWRTkNz-{~=<XecC%GO{B*<hr2caIJ315s-(1u(u9S4D`bC0!gqOALb6ocO|'
            '`By1Sgn^aj2VuTw30|jizi?l{E*!<<S<wR?03N4U_a#QB7(q7-cY!BPxv<GUPF*Y;*y03U9txbH-=<dbl1s|2uj3f_PefUNRT^&I'
            'j%`qqVYbkXmY%dRkC%=!TB0?U6f>+5&hkz$;ag|4?1IoJMMi3L6hzBhXapc8=!=2(1^|`7H*{Q;saUu!tTc07kRie;!=<Gr-`yj#'
            '$082HF8unj#S3JsdG|RyGIP&B|Bi`D>;M2@#;xffPz--+YArw0l6~Pb@~80`(G|LxY>~gThjXVBNxseHQ86&NK4ZG-N>Pq4i3Iva'
            '8Z#>|?lVsWg^f^v89uWm;Yy>iu@FF6tOQGN)ZYg(2G1q>0neg85C_%o_h%TH;nbTo!|7Iqu^A6QALpNv(dsiy+!(-W0I+Sq3iU7G'
            '$pDl&vDiPCc6Yy)sOO_BBiN@|Q~EZR;alN;ZTprNmU?<#vqVd)ShK`GcQy7@5Aq!ctH>Kk3X)NE(lD1lz9MZ8U%wDUT>MGRNU(ak'
            'pcYL^l7}<`)Aj>&DAtA2Vln5Ddg!JbPn(LXN<-gYH1!d5ti$ucpbg&x09|CEj^$IKQMOj+1H>GMaq2f!Q)jCN=e1e%^H9JFLC(`m'
            'AN3Dk_s^=5O!G}4QCFE<8a+}EY}Z$1Wlv2P>4U-9!Np+czJvq#76u=_IqRRlIXyxb;-{sz2dD7n^!*81zo5b1kK%jrd{^_Gzu+{E'
            'h!1Ga^A5AjVC?`Pc>VL?;JAl%C6O!eeK1>jWP**{M4Q*vI6B1)m8z?%R_mb`T`(8k2w8))t)joeoxavnX?9uhrt4+S>xSco-Ux%3'
            'CgK|NWME;52?rNkcf=rL+LyAm11~s1_@TJji0;nS?RZu2ESv}|*>TU8?WxX_O$9vo9usZGhHJ3O3dbIhXr`HaWHqMzl=OY#uOz6d'
            '+3ImA*D!$!EhB(Xc}~nj@?DV1TMD0p;bH<_F5@fXRg14HlwYA?lVibuBTg%=_#pSY)2J@6uP!pWyDlDe4PR!SOrCP^Fi$jZA4eT!'
            '4>LniQ<`J<AJta$5EY@!>w}=nj|wA_t*<L5NUplJ?I~X>RU6ghqapxVkBN>=${!F7rFhb0?GFD-?WXyb^CU_1u!Ah?a|~~sY>Hhi'
            ')M2a)^`TS_c<d=7RXMb&@oWFAe<X&zH^+mM4q7>vWBC6_to&?-F^rT81gka9mt2T(lBcyP#}lLHp1{;M879KPLg=57`o>6qV>=VE'
            '((Wo$L;r;elO!nBg%j#Din;PWCn-@Oca_3OPGVMt+=Z@OU{fagDxd5;1JkF6;u|ko!mzaJVzB8$PEH7MHD)yoR?=8=rOAGLcAR~F'
            'P&N*>=V8oXdwx=kwRbm9Qc(Y&eQm`bk{$&)`6@@?*Y-i;py1CgQ6AT%sg}WtYwgxyfKwyROR~k|)HmtCeoKj?SwI3zUFd8WpkGTV'
            '&8U5ONksNcG?`N29wjP}kzuP!e$>c^^{{f}vn4)NF<N0&mHod=*DuWzbEW5wOvbkAX|)PfZmqZ*8j}6JvVlgoY~^1sFo3oSvEZz>'
            'U(Nt{1gfyZT)$jg$L0x-mzSa7Wn*_pK<k=-)B&>P8aFx#l@jZ!E@<S46_J+fxsA@%NM57TFy*q_By`xVXw5I(d$M7$`M#1|+Fu7j'
            '%Bb$N(T7UYnvf@4Ra1E(P(P2nP_|3nm^I^Y8s)W3KncjGK^>dOTLEib+h`oeA;2(l$v3sW9kW#JeE(Q;BlgdwLSHh&07U`TElY+F'
            'x;p0~X@8SO-P&YeC8=YsDx|Sy36fjqY5gh!lB;QAemJbO$BgrZLse18zH8KND!g>o_0HDy&K~Pc`HSnnJ*ntzXEL_>TCX^b5Z+(8'
            '#+Z@|70q=nDn#<ErcM4-`P2C+O+bzfPyy5{Ku;=bV|H7jS-WD;I?$a-k6d|%ueQpO^g61Vb~TsAt@$%Oma8apG1h+!CX`za=?=Ge'
            'eF$wTNUY2bCn<^}{sk{SR#gP|Qn$LI+mTgen`#h1huxYXs}B2B$<QM2lg&9-QD$4&!bh7}G_l>f#B<cZsWQ6BuC-TQ)t^zl9=-S~'
            '3XGg2PIh-nehGi>is!p^ZoUlfDxh7l8eJ*FY_!Faua$SX{^YIWkL7XJ%adPYeP6b=-?OfATI|nVgCP(metR17b>Q!{2#`sbNFt{('
            't&j>^qI$u~$#b_di>Wj>y~lVLHiR8toDZpd!^lT}Zqx$MIz{WCY+|yKr8Iuwf2WZy9+y%i3uP$R?*9dg8mMo-^v`u6E?Jl*rz$8W'
            'q2U?{rlvudKZPZg9hGSSZ#h~jyhc{=CF`GI(N=erWtrCyvCFb<Jr&1WI2qnercLsfZ`M#u9mvE6k!%V3F$o?mDby@VW#k!umtfwi'
            'G#Cd0c*v|#%;3hsMHV%tiIy~6YWNv9HGE@VPl#zer|y!6dv8RGt&hompGA$e8hJ1NBnPPflFpF(i8k^<G)F-ez;XFR3sR|ZKab*-'
            '?t+Rey&!{DdwFX<Txf0NHd;&yjJGJ;kr-r6V?nlx$7BLvSJ1NBoxZ_ot~2izPTjtcxXs~KlQ>*bf?*ncyK409tB0LWF!%#Hx)|UE'
            'Ie6V-=9P<?uV0L&?zEoyM(de)vtTB}$AU)}#9xqWn&X;|jWuQK)|6GQDU)leCgb$#x8Kd|eyAJ%&-ne%0TUN#w}ADZpE88R(1L^&'
            'sh|x(k+(>1<7`9o#%GZ3YU_9@yVGZx!|RO`kK$k5haRy-eCD=3Hea?{tUvj+Gqv@x{l_)7$Ey0&>B*1P^{La4AFJ$BrzsZu70eI1'
            'oAtc>wSoa9|C;sUs&q%8%ta+du4K1oaWYP`-ziTP?TchNsg@<b2RU!Xx!qbKPgPctT)9S=c|;g(3EOP5E<I@>=2qT*%G-7~yTzop'
            '0PAk;9ed5VY6QJgIQY7&_}@pF>V?Y3x!4a~vA6ZbG@mUfyDf5`uc7~BHQ8WFEG1OV$1+Q<)9?O?Y(6LtTZmY#{jIS#@<LT1L;GdZ'
            '{?pmALLO$--zwRO3*au8`_|c6%L=lVM@w`KZ>SpR`01iz)u`9zOZ~K=wq}Yp9V}XrYAkQ&5SB^201a4BPM$~`+dP?@vu3lU;d~t$'
            '+n)yTRei($#CPJ5Ya^2P6ls$(2|gxO5Jox`sn69riEM7Pt8isUv*QM9!hEz<iTd`x3>iykY+~bMok5T<k^Ij>QbOzc4rMeY%bw{{'
            'E{%HKD#qbd!;_*XtFT08JgTwk7F9o;YEq1r;gXA*VEU<N&I@>gmK2N!gRf6Kq4Mp{y^oE@tlQ{b<tk|D<6kS6+uHkQ$>p^hlM|h-'
            'O^Jt|zIowYHEAiUw4_l=N;m8pGFmB*s79lLwNvss99`yr0U3<4m$TD$q@(rZWa)I`Hw?p=_O6Qz&pVyuFOIZRH#^@cen_ry%G0f7'
            'c^I&<Os+s>6-k4Mtai_iZ!jmUZ7=WTM?gzbu5s#B6pU!6_uhj1#63y`SZzF08Kv2@CyUw0^<vETqFGAQ5Tw2l&tBn0gnI}0Otvx1'
            ')0c#R)dD~oi5t*3XJcP{j1x36OqPkz`MI}!G4oavOo$#NPSF)8KY5tYq+Euus0mUMredIPvB;rXVyP9Q5w+u=$~s!vk}hIv926?*'
            'V_Mna+2x9L@IZkpc_C7Az9Jx|_+~;Sa)A)|A3Ii|LT#Dw&~Y-QwB!0N(=5wGzA;>+w0!?*TS9p(&Z<tM?mu>a+F8VlCO0l73fb7B'
            'Mdfs+9WH79^i-8kV2UKYcUj$GAC`U7iFQ-%=aCu;Us2^jGa@?_>Owo@y1Y?sc}f>VxmMVGuZu5G4hA(zEAG3KCx4O@T|T!o#ZuaJ'
            '1XDMjSuo0f=4-2lOt<=R2~PH(X5|g^MwMXF-;xt2SEE9<Z039%-nUj_*KoG(N&rUZOmGvQGaE`H81-opffb9;qq?`yx0zR|KR<jc'
            '^Gi4#BZVl(>6DD(#bh70C!73QAvF%)%)`df>n$TyCbDOGiFU|}GVa_&rcIdnD?69fzRr*$ziYN2=ddXLySFd;e|WklgO>e}GOJLH'
            '96&$bR+rcxUS*82XdK=pq{TOUb!t7_Dkg2e#bBr`Ee}d{b7amv>#Yj5mK#n!N;FAYn!F^*FLscc@fsdX?D+)r;Eq#6wctk(86&E~'
            '!?;m*W<0}#Vh0lI6vA8hPT3+#`3??}B{22>eAXV{dPz#*U5spMj23fs5=nGy7~QlD-yMsarI$=hRO8$&HJG+*mSZwd4LcPR_Q+fp'
            'MwlJ9Du>w>3fp{W?zP}_MsmG7JK~x14KiSyQBe)9;LB<2{STyA=$!Sbcq;z&TO_V0%Qk&ju^b9Lpc)=IXF=kr1y4*e2>6TP0trZQ'
            'sZP*imf@-85lri3A5NdgbuL_$;Ll<o^#fr>67DE@hGtuNEw8_->N~Q??QqyXem6Y-Vt%+0Gb8=iOn3ogb~y<$QjultAk2tIM*w-A'
            'Q9%e_cefG3+dLuo9>s%@h=gG(xI36vAj~MaA&76f2+R+Pkr+cMiGQ1hqLX(mVM1kF7%->5fQC0X?4NTb?CTdQ=bg&;Sm}p?wbLl^'
            '$8rSA%$8h-Jhl{#@ogN8DL=_76}Pb3R0WE@N1f!l5(B!jG?I>lK`M}eSWO9tF$@M1k5eE1AZNLt;8G#&XcQvM(K47&A8X_Qs5z@N'
            'UE-ILV^g&qS>V$sX=7%OE9eJP%ncbtbAJUyP7Zy8VQ;~{_=Lm#OM-W_{w&0*NAOHmDK<uT9-M|_mKjwuXsUD8P_VQU*_Ebp<Bw6U'
            '_$cJ|()hH$@L?ReFd2zu{}4Hj;8Zw<Se<PQ&eXrDc#!qpe4SSrst^EFt0HAK2D((gueL0Bpp8$1z+?lmDmw}SDEaH5Foef(%4(o&'
            'UwL7;QLPoJg0-p&!&A8@dR&NO$bD&b>33?IS9FeISFO)gi>PuSU0>l@>o|?l{+m<))zx+9f1B|3Vf(5g=tY7h3g#gzZQ%={OXo6S'
            '!-8P*fGbJQy^o#pAWoI{sx95E-CE=9FtL&-TdX$Rl`Z>!xfq=F58WQsUgIuS2|QUU>oe+UnQx7<S~Mz`)1u1&YxvrdTe_W9(VXm4'
            'S<Qs(+UNB3OAQ9*4-(n%SY4q4s`Hd-l{tv2j>KnFh$fQDpUDeyRg@&s_*$$bU8tm03TCb^AzfE_y0WbFXV3HT0@NGKeB;{GpAW|B'
            'D%P7!ss`84{#M-eHBHMN9ZiY#Dq*ULXI91R-?eTvB@hP+)!B#^Z2|xp474U?7ugE6DWT}Yn9r+PP;|;y1!Tf4y9$5K%K68Gv-N2F'
            'Kb`(^wK!8%susCRwFS8}-TvOyR>f1L8R6Y*SqyiatF5g{-fHpGEz7>Cc<Yws5n>!Wf`j$MY+K^T`ggGvz9SmBRl%9%FY}#d)3Sm)'
            'gMu?J0_Qf5N2Am)XV#5kXPbLBZiDei*BPurZ_&haz9UJKPHiJ*KuReH)p=21*=o$OcavzAjD<J13cUD1s`_l#l|>pcheK1Vq^h!f'
            'DJ|>wl-$<Eik_ZbV7oGhn66k#uEx2RJMyAP+MVWJilOg1hMsGnN~PfdFdhwq5>SIfCP|AJ#G4gmLG(G}X}hhn0dU6FLc(~7&<%<x'
            '<qVj5u?DZ!+?8zZ&AO_P(s#@Vl;o*^OaYSdI0;5p7&yIb7HgJ19uT@Fqbeh$s(ogcgwO5YjUlP^pY&2MxA81)Up@S0++<A~WRQS%'
            '1cSo6{KL?oxTr$g9^WA8a!iOud`VdNo_8+@g~W1!!J{zpHWRAJw#<ypD8}pDR-<go#Iw50LQP4>A}aPVUl1nFCCd<Hkp!_=R(v>M'
            'gWQ>_#zn=S%!*67JyaT&Tr$4t&r#*r4_U6);a?>K7CvOGxrJ?`%Z*%2(*{RMC|epnw?`@jwNt^X8bq?E{73hq5=dO>!=aJ|Nmels'
            'S@Z*+EVOb2Y-z8~KvZg5QKFlcr<f8ove2_3j~}-y?<Cvt^;TNNKgy6Pc)IA}&>%C;T)RMF2|<ds^D`qrfO?by1?2s<%v{5!8sa<}'
            'N7TD+VK<CPOQZAD_b14WX;Y}1_PJJ=BnK;}Q7?|sG5dEv$$s*<{7wwxL0yUbn6#+^!KP^Zy44FH`q^+gOF)C9S%@FYheKj+SMf04'
            'T9rH=OOn@YaOoK|p}adgeKUAD81@hGua`%s2XBu+ci?ge2ZLb`b;DA4f`+ZNmhLBGN)L^i)aI_N25;2dod!3iS+#tBr%|30V{B0V'
            'JY4~=oV&;%umBrSSR-4yRgZUr!MLa%NV_&4Z*g|DZk{aCw12+DWpah`YhL3u-z4E)UDs3|zSi~9L*R_~1~@jC^1~}rN*9QavHa$a'
            '$l`^&kpIr$UyWsaS<(^jWO}rCIt|9aihz>Sjq_7+vxJ72-ozX+y^WXQgiMlX8w|2k-3^I7OaiBT7jR9KnOUIojxW54E(Ibmw>rug'
            '@jS@#$~QqOlN*v`b`RrcF&aJNL=Fg_(^K*&;hrQQQ*RCu1?KjExuZuxG!B;&jLoyemd9h}O<M%-2a9fagY=M>Vklroj`!T>vZA0f'
            'I<hE%-+Ggm94KKUXQ$BtxwJ|d9NM_Sg&GFE?*uPh7E6?S*n2lng^p4m*bfpc{thNB@?TI}^NWVT`a3|v5S?5o|34)sBO-}La}~-('
            'qy&cZ3?~u=gYtSfz_|37vxS1ViH8OegmgxFK(0k>L$n=?NE?L)Wzg#Z_yK+MQ^Ot>Gz$7xZ)t&cm?|!90-L{Fz?X}I;l)}15NT_^'
            'Gs<p4W!uYB(xcCVImlM4g>-}Ru;u~RTHfL0VzoVaUJZ^xx1Omb<n+hn4sznJ&(QG<rY-FWk50YDO8!ah!h<r{pHrQa{OdprV1H*J'
            ';NL1vI)<F1HwdUBpNGp&D$N>KOQ9q;4_9hML^zAk>~QvvCcU&IA%3qI2)SFjlwA}VIuN&})UaJl1^S_7I+?fFJJX#;EzqS>ze|*C'
            '>@B)K;g2M`DO7nfi#gq+LRGAMq{}`*w-<Sadc4_zCnWN-c*ApJ7qkDs^s6u#%HYqpQ`azwYx7pun02)wh6>G~1r>xsS(_#A0-_OR'
            '<E%0p!`^l2e}M!hfb=r`eQ`{f5CchkFPcxp_umT(u8xDxcRN3dA8@b^kA&IU#S?vWdAvUhBT8CrK$G4TYS_RNb)QZ;uU~uHO3vuG'
            'jUW(=L-Z(>ES>*wryixuRSygLcs+yHDxZ=NzTTDY*SpjGHkZ1OdpYS`ZS}e7%o*m+zKEH?w$)Hi@^dJmp-$eK5z8*Za<tD44yg+e'
            'sT<FX6InB)bXRhf7ux4KcO}ax)%V;eOHpoM6ua08LzDw4&vY5e5}wkaBgMpPc_x~C1X_lX6xjex0xblJuUzAl(U`^J7#x_TQAHUC'
            'dm*lkEpM8g4L*(IQVG4uHEjEe*$L06vf%MGmn18c;XXy77EY4-Tb8{=E(jtgOC2w<)-x9I2_kg3!h3Ws6Cb3UE)p-YQ<<r91lWfq'
            'zyYht>aLO?%iNPb7K5A__A<0|C^FR+J$bH#wCICGFgO?<{VLA;f4SgnW$*`z3}<~16i&{SKt`nugpBmag{7VdJ+5MpaxBajWqc3U'
            'z=k*`^elY&cR{>lTmY3|8AK@ebEHJe@CxH}^W>_;nH$-XH}xVaVx%Nz7U424jBh%6d02T8bMxWek>p({Tc|h_b1Gy}%)eS(d6JXd'
            '=LIHt$U{9Oh@n{LfKT}gxoA>JCl!$pDo<bK+0xlHYd<WjO75`Nqg?_NCqy=jcp-v0*~U<dA~$R)FOw;-CXI$F4%(97PGr$}S)3EM'
            'AO29f3DfX=V$XK>cAo7%)!r#N7ouV>(m1DGyzYUh`A7y{(0o&Kpl906r4E)Yea)fD)DW!nWg?3x`ItX`_@KvByG^TtfeKk~88=Ua'
            '5y&AlZ8xT=j%=z=y(*>!-=GR%c+OAy+D-n>mw=RK=e&2^w>?qiwwGZmE8Hu46MW!&c>3<VBNHu?CDoUwcSo0V1;~t67{xF9=i?F0'
            'V5L<D+Q@8p8Jxv!K7QLRxgug#>SI%Hlu*&{Ldh`rj-O<16uZ@lCZ&Bxu|m|D)K5eSor-xY7asJTTlF;^+kFlHXY~o43&%)Xy_MI%'
            'Qyz6CFT(s)x6=R3&i14Aw=O$?<bsm^Wk^fvM@tDt{Ey9Is~w*zpSCK*qHg(7Vf@eXUpB%FrQnLp$qGfNxx7;-h<98J!A9X~ubgwO'
            ';=WEf@0V_1j8^I{rZf;1@{QIHQ{Oqi{j2J%-d1#1E7u!P@ce1;P?YD98kFIncGyL~Sa{6_w~F^Q`QI9RXe$70NHpcQkEEY7f?WAr'
            'k9<t#{&+jCF<WYe0zx!ip(u)J^g6qh_p3~Z*4?2Nx5j!cEzIWZv{%11rH3u+2Jmr=F>lZ5jcEzTV{)lhK{gRkrk7-h<QQInZ%uPE'
            'PVVug74C}*H<8koLB}n?8_5^q@!Q|hyv&Wd;_Los(k#?GV2axkPb;#}rBtc4l$r-##Ni$Kx|0BpS0VC|e4-_LTgf%`TdPPKFE_u5'
            'Jmz%y7|sAKe)s+un1fw7`q9YddptNlADq12WM4O_aQaO*g02JOG$4-&%l!)^@HrXZn}pTQBdJ+)H!IyvxyO7BhnXPH7*@Q6VVIQ3'
            '?Ui<)DLYG6u_aOEWfRZJbhny>G$>RdT)FZxjeK*l0lZUFr<=1dOkFr@6$x7FT$IYusdg+=b$n_eQIj2}>X74XMO?fbxh9A<SANBJ'
            'q^JQPNBUB8%l@`JE!EuOsxIm7Bfa$6b;6y3BNLOCf(6FP8jy?O)vxsqS&x+!kS!2}kbmo?jEgqndOy4RjWLH(h&h|CG4qBpd=v<1'
            '$pTUlBF^eVxAx|{STtO3D!s2s{z&%pa~kQ48?s$p_9~Ca<`c6T0-IY*D+@*-<!@QiFFJA*XIo5bR-*jsKe<y>gd~`*N_4n9<+-0y'
            'h}&lqZ~F|}5u97~X494(nyBtv^XNo7F7eA{d}YY%+6KMy(8f74FvHFTz<29)IYPgHVt-xAT}{CWk~z_qJ6)vN`5)EeJ~(Sv%`>HZ'
            'FmkSIo13Bu8p!;6SNS<M)^;oiDfnZ951GPAvmR`(FMjEt91Y$I5PA*=``Wsf`8NTKI02%P1s(BvdFMxAFr#=R_z>RMdg<JoaY%fl'
            '_5)J}(@v@?WQao&GX`gaqvQVJ04)#`3Yn3c^iVz4`mLzePu8=Tb>n@hSXP-gCKu7F4qBcSrgt%X)2fw0o&P<8>(7)_EB_xJC05O5'
            '7&S8!llUfCQqqW957QrDJ`^&hI{T%ZD6(h$DQf8BlVU&)rd!3x3ITebH^T}xowY~M^xnMn!v*u&ZOOPmW88KgL~n)ux54i`i5d@e'
            '{h6!0Mp@&3>LJ$L^5x*@2o9_Jmy6S3?|;ULt?&I=Cs`&WEXwQHv-EJy2qk?04VA@b|JCgLpM~_~rK!+Rx+AZZ#GSwLICv<JD4HWl'
            'T0?N81mI&Os*wC+0GZ${c&a;=!lF70d3u*M!%dbS{V)p_m>4_pGElo#euPzDftp4BTnZE*N*IBLYlh4GS>h!)1(sKCe4>oG@oOzP'
            '{9j7<HN=(k{@E}6GbOML2gm)>iy>E{Dl6@Os&S{NoJzTSrRI$PRXI<U$pLGpZXs7E)j&YaaPu<Ux$hOad%*u!!?JAdsbv^;D!~7R'
            '-seGYaPmv<XmIEP=DoWZ67(ueLVpr_`!!!GwXKDfzMB@`ZFCb%h0p2ume*m~m|E2)CoM2natgZG-YTSTlr9@~=6z}6sLd%<iofvg'
            '3x)AEGILa#IAjg!FT0?$R!~T|w3&&+`7bvw*Ax&dUr^;@mi%*LS?bGWs4tP79;#vZh^(ZV$xF*Csx5@}!OB+F6(+GS@4m8OMRHp^'
            'p_C;*74Kx3y^@Kw4iAt|?p2Ov-`~=L<W&p$hL!@PjCW<qquryr;?k3zWI?nJKQH&&dspga(z`@lySmz{lLm@+$t0G%P{EHixg<V$'
            'm)o?L+gA@D7<qKlu$?0=Xo!t`u4)Noiy~dl#?jcH1&kZ$c(3m;Q8sIAJtj`W^0qb;i5rcrZy{fO4Y{gDp89Mm^!qhFoBWiSjX{iT'
            '#<aDOw)ce;>2s;kL;3Rb_iS^GT2_XR=_XUJ^=+cHO&i&AZe8HZ+QF%p1enjK`N^P9+a(OyzH%O<x@3ibH^#v%3&h2*hZkoTN5{Rx'
            'L8oElA^bhaTtMQlo)0beu5C*{J9$AwA;jcf`+@yYr5oLO|G0ND92}f~ll*8tM+<XlRZfXWF2fYoYa}$nlJ-E-Qo~*L-_B1@WERC8'
            'FG;)=Ik3i4iYQg_#nKZujptOWC>8gK$G&kv#$+{4DXE4huCK3g7GjD;yPb|W!yCC&RJtRpBDb>26?^W9n<cvM8u1w9)*@G<v*gOZ'
            '%XxC;6=2o(nvGGMa#3s<G$SKKi=(k$0+k)dV4x1i&NPz-tgiX<J}%Jt`M31@X6J{W9e4(Wjq~+E+jH%XLj#m@{L#}62J<YMdye%*'
            'Rr?QPK$-z@t$-9)n%fi(b+hpaV{eVWFB=YBPzD&X*4}gSM3eA`b~EIGvB8FbA|PcRqsJ(WZ<gq@BcbNApznR)i6uOf2R$?S;0mYq'
            'gA`0K*a*7)saq^oxON)G=V#u+i{{=uNY&oi4m@<=jY`@$lS}><eE7=p&Z)nZT{oL4Psdv&WC53A0SFy10xX`R^o<BxeP^t^>(eGn'
            'ldU{a2q(qtHUV_wTYo&mpN^3kl{*^LMf6?`F$$Fe({?1K$zMya`g6;{pM8UA^62q^net74e|o(yu0J&}5d?VwHX0AtEc&K0|EP@8'
            'gZ{AgFvL8WA)Zrl&GWvdl-w}z-<Nc5;wg1(MjrZbgWNSCKV1m{hkc*p(1EW29V(@|E9A^na4Y0zUB6o@lN_%+@>BS~pe3+GJ>J`4'
            'kf54Q$8~X{uKOU7%gS6@*exsij~mIDKOU_r#IJ6SG8eZ%^Q7=+X@JQuXv}{}9*oGoI}8!j>c}OvxH9L#4UGIoX3cea$(ik>=X}N;'
            '6!~$!c+Nmk%QHsk6*eBh!<b@F#G>CxJcA`7S~N=tQ(tnu(z9ay3cLu;E_$ELF#048#r7wh?!z|Sc?sZ2)#tj9KC4u0<nunN1t|Cw'
            'ecsNBA~jDLY)PzLo^$eNd7OZlG^j3us6;_4h*p(!QFedPPn^vxYlRi4>02OBllcY`TrbH2bmAoY7%ordu@!r}-zbOet@q@|=L{~L'
            'oLHxPc`TVQOzk!dMjaldIl@d8xBiDoa0ALj(^;EpxJhA~qgCdo&KKnz#u%I=!#Y|inZ~N}XRgU=Ofms51;DvxW0_6cKiP<0SQ`Py'
            'TAT&IDJU;Z47#!b$c^D^h0*v+6v0RCgV+;OmyO_H1L3#@av1t{ZW_w-g7hi?wXDfGy!U0x4roY|=JOwaeEJ+t9BzY60(crKf|^FM'
            '18w<zd37=yhrHRA$k|i@2B^lU3|HT`gcXguAP?(|>u#(&dTg$g87``dj>!lc)#TBP9I`n>x^h5PD>pB2CZrgg>!Ovz>TFAHx*TDG'
            '3t+l-N|3c+pv)ES?Ggx_>n%`U#Z*yd{I3j*H`CS2kFN_T^-44a(TQUsS$%G+L?d~0B}YMm>V;N(9b~GM=pDiU4zSxn)ZRf8Vq2yY'
            'lg_xv(m6Q^M_5uec`T2z%x>c(_{f6eEnqHMx;?`;5aw}lcGP0k4Lv%2%~hGuvk*sdl6ht{_CH0@A-YL9X^0o%!B~lV^q!Xyv>b?9'
            '>7#h<$c0lem+3O%NGC4DB7CfUy3IR-`L*C<XjPbSE(@2wwEco|WyHKRvOuw@N^?~)yw~Ir?Jk0akLlt%YPaPN<D@y)h5T#Ep;d;V'
            'byYkXidH2*IGs9Ao>enQLOD9m8jrI@axGmm5E?0^`NlC4YgjYo)XWli4Nl%<fo^S@X4}?PNtKMP*(vXf0OpbTbN}ND*a}Qw1L{&K'
            '?;jv?JM%^$5(NTusQJVb&Dy<jP$0t~l2T}ho{<#>W*8pJpwurJnDqf+<*xk346p*lauEknMutU2hmH67!@3sLA!{|!XXPn<X#iT?'
            'NUGVUv1oPUd}dHu%~<8872{RJuvHwOVk2iwj2kH+<uUALj<YrjZcTR;GgTF1RyfP+Le0#-DW6CU)u<0Y12*6>Od9)n9daz9WK{&O'
            'E~r$9nJa(YeBZ*DI%i#cU%7uVPu?ccf8|kXnmQZA2tGa-w=~DHL@X&7;bs}Kztx0Icjl9FOmSsEW%(PHFh->??FYqSZAx#>(EfsB'
            'jJ0--P9eIpAfV}kReoh1rNjrFGC=nv*mf5_6O3_dU!C>)?r_jQa}RsN9#;Ar9Q2KT)p6cv%F=F5YxZ1K1|w&)iGSlKbj88-!G^sz'
            '$Ago@!C6Zc23z^r3@CXjlgv+$HL2K5r5aUBnR&R5WL=jHy9FoHve~6(=IWX(f5e5-<Xyae=#>L*X&_u?TG76QOm^RgnsiZW4fP45'
            'REuw*z`yg8cFs~!bc`c3XoB!P2Y^met<bknk(*1!9(wi-f!yHAf-1OS)|b=wCuF{w$a;6g<2xX$`YX)pfQKdVQC5e+$X$j=MPjF_'
            'P`)-4q855%%rb$mw09CG%|O{*u?859Qmo7uq0hBRzlkM+PPvOBhs^uNN#{tjtj*WBQR>IFR;k*Yk9lUZZng6E{r3!DcXsb3H>ng~'
            'WsT)-O#vug6kb47zb~)Usm(rEHp`Tobj#0lYCR{ft%)Z0x->ynA~rmbI4ZBhG(B#f#jsnE#jr5^wDGAt-otOvZ_y|I@<TCY9)|2y'
            '=MJq&Nx>^DB2eLPd($h>u>7bBp-NBeadS-5zgm^y=A7=1wy9H-Bb0ANyZH(gRJ-a&OG2#6|41xQMoqk9f5fb78$}DwI?Z(%6;U9x'
            '#}Z`OsveC60$Huj3Vz(L3Urjlc<WGF^(Cua<nWf%h^bbkCF|17ai%#t^_lwJD$xX0oN^hdk#6H`9?#h)1GRNY5mYwY?`6kL5`;~I'
            '2p9_t*ya%hk0_d47p>PS<kak<fU}${BgdIP_Jcb=6*4{>2tfjWOOIK)bHfyellVRrJ0cweq5;|R$RR!*gS?umn{wniD)<&vqgXUk'
            'A0}{gbo$;s>%Dg|G_B_jPhBl96%w>o>{bC$Fi&NJ*9pJnURQKuWWv^4c}8Q|537oxp0$G460H{d)`gx@srfU&98d@3+bMJ=SzPLO'
            'R}Q+PNg2_xaL3?ejpBAU48Hb#Es`+w6(z&WK))AD<bLKOYAngKvTwKv@;df&_du>`9z=kR(M@)1Z303UGh)jfP~X7Z*V-Gw!NTCr'
            '9*21edYb`#v<zpi40x8wS$_xEZ-blL_FWt<=f0W$3UkaU(7!An&>#W|6)I68LEnTVz8Chapj(iI{tlN&AiF3TVKeW>*x;6NjyZu}'
            '=zGy}0aR3KuiA@Qwl}6pchkTh?$94Njy@Ab>bIr>#+z-<U66P{`GJEai^R`<@9w~ZgR{Yq9@>MVlMy+b<DV%xBMADqncqomphJ1%'
            'l)Dhg(`l(Urc5O>Z*Igb!KG?>n*oL;2C5z}{ef(#usD*ZrB_KUfBZ~{Rb~C<`vQI<L2{tD%S9?dVJ4iD$*J?+RCh-VP%xRu%y*tR'
            'KL*YrqUC%9V8!i-7cv8-T7<4ug5}S>6$>=U9CaB=JnmtiU`%Nhmr26q7K2D;!Mm5@*JzoMrx-h*ct$-&n#iy5a=0A%cwCZTOp=NO'
            '!VnK;N~zN@?Qfv+yq$Al-ix!NbY~p!E~cbB$;)b=Z$)P4DIlk#wbjS?p%x7RFY@b7op8*=rV>t6qLS&&s0VkselYuFsw?)=+ook('
            'H66~Kx^e%$BtX@SE8lKBu8q^|I&VF)S}QQX`#ayu3XY_W*9slI1OtrEdx!mlzStgNsr6_HXo0s@x88`%FB3qBU>mQVmuPm;S+5hz'
            'ht$HI8Ch*8Y|uuakMOyOFbD(oBC&#BSZuR@aNPe}!Ryra95vl5D|W~9d%(jT>ay9|*D^J5mDowDTr971C|+K5^3fIBP*n(Cv5%35'
            'o|n@YPdOp<jlNR!0&)FXbw5_U#SY)V={E;>9sS-PT?_PmN=5TJ|0WkVfoa_Pqs}r3b|mME=Uxz9OCeHK8@as7vvF%L(J|d&Qd-VD'
            'Rk>P9$-&EnT&>rGtT!W}4Sdu0BUv#Jiw`>MYlT=IFKg?CXg;k=pqPb1${W)t74KAU_SaW<v)l{hr*sW=^Gm!P)g^CWhm470L+A<*'
            '3k7dr6Z2;98f(uaLAfdpP9NLCm+&7<T@PU1f4r6(XQ}j~jyIV!p{qTz+_W_zN;sQ+{AN|i4`Ls@wWiMxE8L43jHbo*@~Kl98YEFn'
            'PP6u;YkIl%q+{i<Yxt<lvafnw8x?ps$un}tG+m-EV-;_b=#E~{id2FO7tE<v@Q8B1j%7+<%&1D4m+w8GG4xqKFHSBez*lRRAfQ}r'
            'P`+vN1A{~0CsylgI8~h-#?=C#su3NKlRqNeE7hV(`!8Q8T%%6bF3SeW3@wJkb0tR|I{PRI_22a{q*ZF&+5{^FcANn26L~a|V_qVp'
            'MfG+6cyKasVMO<r)4@UiT(i8!-~7exDxvZ|%kd2VJ-GD}k~bS~@fr&LedR|<Ky8lviHHB50>8t5d$F&}>^2Snb%G?hjj2;_nZ*(R'
            '?1MHPdK2z4@D_f?KQHDZSaC?L!<)b#d1|Ob*kaIBsWWgv(3xc;KMF=vaL7wN_zN^9nwAGGCVoI2dVpj}<RxmXUP#eV`05<-&!d&R'
            '_QuKb1Ft*89tr4LvU`w_Pr~1TXGcDMb2$psXun{2gL&WqC#T-$fHLFT5q}#02to1k=@aM6(3^sQ(b{shRN{TT1sLQ)@U>nY^-f-2'
            '^j_23ENKtV8e8A(i^F)lWLvwEnGrYGgt#xBJQK4wA6sJYd3zL)!4@-6d-vkq!Lew<?=hGA^Aq5Z_>uG8@jEPSjHuLw?(XjHuKD8q'
            ';Bfc`-_XN@H@%aS{t=?gUSTpg3$i1AP_`ldz`^Osu>V(B(dpa%2?t<9@25}bL(;yls2Jh0OOAhbA+c-;=zU2qT84j>JbwR7<=uh^'
            '6}nP=H-79Ec(LgONewn+Vm9mNuE;*hqhocMik5lNnbKs+LP6PglDWU_J6SP5RBxX@>x`a1oA~5RiO;LDyAjFbZKozJn)lYD(H4kv'
            '6!-!7<|)uKLA)c;S;l}cnk~k2H;dhS?@po^JlGcssXU~hF51hxg>!*MZS*n;)2?|2YFzf@K=;8UyX_Va5{D7s7d0n-pLmN@A5lO-'
            'Pn->2z1n%-`=y_sRw-B;Ctk)NNiR7JZNLFq%#5!)u*z%9AtsA=j$i{&8efFo3NM%08%Y)tpcn~%0orzck;fjWZ1+LJk%71XD$=!s'
            '9{|-DtyQROOAkI;6R6G8DOJD03Ij2f2%|aJ!ZfxeKvP0M4R-OphBN#$!I&5QbG<OQL5X@@ONI#IUjrf85I@LubQU0nG#h^!06~8G'
            '93GS5c08x3pHTxLKAmR*JLDCa3tiv58NmNHkixd`rdCbF!k|{Z#%e4v7Spm80PFe5vA#dzV_mrjJ%7tPnV0_q;=3;W7)+-uqPuDE'
            'kq@FqeSVHtC+u>KI2uH`tauJRIKYnYm=UN#it$sFm#`?&``*#p!O3ebUO);13UU%{-@o~l3@JVFYVg<op*Vf@>byUsS2g#(Tn8h+'
            'BY*a+jgwQ00}_ymYgQUPR86N?e&OkJDm8}%4oR28t8rW|GYJfeU^IQU+kW<pF}NdME;IHB4W#V|Z=*v020jXBB}S&ouHXsTYvfoy'
            'l}-SH=mX!l#>$7+JJ*AgSEtvi_$WK>$Vu|NrqlVtzY&udBtKA1X_C4w-Zc1tMG$x;6oNVTJ#=hmTiN@wQ-Cf*me;t5feFE;Ok@EI'
            '#(mfhppq#W$0+hKi(`?_(F*{?$P^AETD)3F5WHpT%TIHqab1!szxLx=zyOGpLaI9Ay%)}8bw1i*RiTeSCqt3d2H3}5E)67yCtG=P'
            'SNZe@@e_vHAwm-eFm9IFN(cS$#U;1j*WI4wK}ERLTs;$y$ucqyiqRGO<lTR|axQnRxFr3(ZhkNd=|2uYf?Q~I2E)jdsuZKcJi5Aj'
            'p!qDLF*!~Xc<0MBlbco;4WY+)8Bn2-TzIDiArechC(X(z6k{BEjKIX^WYcZ#wsJ^{wrJ+db&R+qSO}vknjzsdOxi?{P}M^w5{Y;+'
            ';W><@w>P<lcoUj#J`FW`7xwT6?(l=4*@%A?dlqflEk+u$#HT|f*eV@F%%Hn_yHB2>Yx{zIA=mKe3PZJGojZ7V$2&Nm9d%ZAfUdOh'
            'z7HAz<D^?DX<}yoVAwWb2a<ffqqsua5~6)_J78!0aN3d#G4k=)0oR*OIl(iY9OjZzXQs<k8O2L=e}b&Q8L2c+7rvL>a>R(NW{h`c'
            'RI<?j07niTqVR1eNEvb{lIr`*+~rCO5C4CA*V^2+k!8Q@S0Km-k+DcgnTa<e4Ku}z=vWn5a!Gc!yjm6s5)j4Va{)p&t>J&)`|3tF'
            'x<Nhc%*;-0DwT+2qtWlak8{sOk*A8IE~L@vp=^{Lz9bWddfpiBo<Vs4KOj23n)WVKqugV9jyqlHMo=ZttSNU0`$6E)lsl|yr}-zg'
            'Z=*U!ZhC<{W@8Pm7rC5^Z$K%^dP^i$qVn>?#tcW|;l5U5-F1w7*y=2^&~4(@Jd%g8(cDP{s`6hRBT$jb9+y|V$kxaex0vP`(Ravb'
            'u`Ep!j_0b#u61)LugyVI^ect1t$L)_h*c!`32s-boSNTKu~xA&i+r2LuW+a0ny0h&=#O1IL6roizyI;*@O$|M#VrHoqh4;W+O7n+'
            '6c9c>FbLjTDSh?mUz}^teV`>cYS8xwM}K(PEe=Hjamr2N6`S^yL}on0v!GCg_oc;XHO*X;y7krkd+vJ0Zw!i#V%Y3$OpXk5uyD{1'
            'X(D)B;L}UDn$n)9T1*+e)$$(4mU-DMPefr#R#z`CMWCq!TrFep_XEyTS*ah?s2BVDc%|9ZsV!k08wC%C`r6@<QA6OKUih$XrD7rC'
            'GXR-P5Fba+HYr3oD>=E1<AVQq&Us;+6%@N8Zzmj-oF2l0n~lvl-Kr3t7Xccjn5c+vJ`m>bikRmnB2b|x>QKC^Z&Xy{a*7!d7)SvR'
            '_ExK`CLc9@hq#t+;o?+Y<v$)A9JqGqE37RGF3!(Bu!lxr2AUxL3o?bI^BWT8S15=^0cIwdCQ1ik5Ub>8p3NvM3x-7!{28cLGEu>F'
            'X`{2mdz#HxS=CLN3pwaG2=>n{qi>G?2g`JIa4b!a7_nI`*fZo=^a_PV`uGg>Ze${$0RAS8b1e;PfBAFky7|pty7m9gUI=eT-HPbP'
            'wn?pZQEF9PL}vj@R=1cpBo{8ZeyDxQA}vKUzQ!)9yP)2)V*>MzDheY#GJn_v_v*<_SRU%qtIZqY!4@hUB$eE8ArY`v`8;Pgl(}F*'
            ')9Yyw^lmX8>RhglxprIL)@VM;F%D#CB!l%Cb+^<pM_L;?f#Pw>IgE5#+0X?NJ_e$ZEt6=%EQ-pRhaae-nD>MVZ$!dtwBpcGB&E4;'
            '4`5%Ze4K=ku;vVnF(M;bu63TAE*8s-a=l5EDL3cz{Oa_iLwmIEgID%-no+TDVMrk+o)-$p)khG-L3HNY(xgHCHCclaL{Ty9Se+FN'
            'XP3ly9GJ=)j*QCR76X;YoVo|^AnM6zs&co07XJ!$23EZj%u9*>UXYT|Pot+%n->O{lN7Jdd4?>O*qD3{(n%tN|2komzRKcCmO*{m'
            '$xDwEAYZd!Qurw1S@<}NgvF9n4(=UAnsDbNK%huRoyxe{*-@27%SHv-atQAC&njl9Os3-jp^c$6)8uA_vFePVX*4)t<1N`MRAeg?'
            'n+YY;9%;FQ<TgUGyOeXD-f=R~bwGkR*&J}p*_0Bwm&152;k0_F1b%fJ#XLZ$mravd$t)rJUg=%xbKUy2xs2$vnN46%*7!rFXg9Ha'
            'Jz?SG#|DcSb7LE)OvSx|cPke5C>auI8Ln2D1jTyT>561;ZW1n!#!(K+BYr99%`iHa+i{dy{La2gDzfxdMuEO-U;Wy<ym<evb9GI<'
            '>H0k4<*-kx(N4y<HSb*!d_+R8D3&U4h`uXWhqT_O=7A1;<+gP^q+7y|Z$b>lna`3n?0myK4>B1tvQWL>q#KY|C!d>c&Gk?~_RBQ6'
            '8Gphx<2nA-szI?0BOOAL%X}qE%S;5sP|nXKxlqK%PKum(*`FHiAvHA-II0qd3ap9&GOdSV5o|A2RbYaQ$_v^fdu6B!Mk?I<)M~;1'
            'LxnKoD`q^r4%_cfPA|f)Z(A&r1TkP#C6x821qiLn9ap#c*?*yjv;d5nVfQ2$jI!!Ezwj4mV~3CQVY6{-Eq%4|g@YJ=RM3Q#4{)oO'
            'wTurk3O$}Ph#Ycwq@YG39=Lch2i+;o4SiWJ9}(q@cli?UQ3H3L!4RYjI`TS2#iu@BA9SDCHKSY5%g~L#DQez%Zp4W{_zTy6y@c!Z'
            '2BOGy@w7;kQu;Z~z4;B(K)Bf-jWf1e&g*8EG0t31Nr$=h+n~K#%*I2spkoTX#>W&RZs%x?<-0!1(>sbyL0XU~yQSRRQNYIUh}d*k'
            'LXi4crU$8!1x`+_&IaF)i+qYPElEUZU#dssJCp{s2Sa;|(GFL3B9|$c6;I=Asl*B`LlFHt-O``5n9=0Vc(S96QSdy<^GzH8G40j#'
            'TSYj@ngT7a>GFjlV@GeXfyHNcw*{Vx8Ab!`wt}T;E<Rdi=}_YW_*aQF)jV9u^k{x#mGxYF^hCvls281H;Dn)LT(pD`Qzwm>a@ue^'
            'F2}Zvocok#EWZu<o&oP8vmeN-;@+c}p#Y(CX<-FA);>(7t$d5P^?+hQv=^`NRZs|B!QI1pkTO{{t!_AGv6?}~Na$+S2*S7AZ|d@`'
            '`J6p6OJSEO$jY_JR<9C?7Vj|;kb|=>1gC<p_w8lpx6{sl9<zWN5cw{gyolmwcyx)>#)3sFyOr+SwKBtXw<ohLr<kcG95(A#UmMP~'
            'Haq@UL7_^`y-5B*lKvCcmpIs(NNh1Mh|VUXwi$?Q(FEOr;-|WS0n2Vtd&kK(ym9Ja$q^1Ssm(Z%udSav<}5Fr&aNfJWA*z|Bg&VV'
            'PaZIwXd@UI)M?mE<D|FB=NMaz#jGKljEO-tb@6fT#cV1AOrB0)UQzEASs|$Ptzl{H5*To|UsvI}uroog6&*&Y|AqOXP$Pm%1v6qs'
            '<Yj_%p)0hgu(uy?Sz5JXD4AOUz%eMa$~%Maj+3Esyaeps;9Iybv0H9d75~$I`}XqUxAxhQVj3&M7_oW%(&p}K!~$7Spzw~?9N2>`'
            '#|0uQ@J__T6E+GI7~iZFVnGIeu6hJd%E83EVSUZ%L=u7~x#*)&iGyIX0y2TJMqSw>G$#U$7*54L<ERk{&4!5a9QOE_efv5$0To4K'
            '$8OOWv6<MEQOZOWA^aqREtby}A4br(IXU_aFbbB_JX4WPJ6bWIcK-v6AaVq6v9A9!tsdN}|D$FPF<c1LsP&*z#(VhH#qfa9U%hA8'
            'AGQ@V?G6v{UKq`+|LfRehKngPaD4@9iK@ymmsrs?Jef{v@8{T49<ZsH6aGTX#y;S&-@fuZ8_N^yEYG#I{66*;kIlvNmF*z=68p<z'
            'Y%pKR4r4FQCm3T;Ae}}8XtQGJGRcCaf@i+OXrtPttTuyC+|~M*`d7Bzc;2w{b$CPmjSV)>u#FV<kZRM$|646wtA&JEDpVGKtosZb'
            'a-~+7HmnQx%PbuKC<_25-l^97gDY5oMvyhK+;Pf_ahfIjDg`Psz4BLqWX>YSSDgrTlnjy8;xo(xraUz&Tqkid!4pJ;dgF0SwEo*='
            '`|M5ozy4(`5Wb*rroci3gys;&4>nh~tI~g`zcjhATY*C<TzM&b2o#w&&DEwg?Wxv^8bgp)po^&BYE_6WAZ9R3uJq^j^dn+~T=lUg'
            'Ym}E4*|z#T(*6Cuqek=#KYGM9k^&zgt>e|}hgC0)Q>a!7S-545Fe>yxt%2GK8YY@3rfUmbvq3TJr)V$O?sauc(B9HlkzyEZg^rx*'
            '5&T-$yCnAQcI(C>ut-8%rY^GMrDJcEd*?fi3=N7q-|{kB!)P%>SmlVBywv47Wrwv&)Y8UslXHoe()bw91vrs0TXQ03OpL4!h-+z|'
            '!+-zUIqSUXT%86#cYZxP{cv$vPMaKFXy)nXu}ho1_IKJt<Lld)p6~W{Tmoy2b=~p%-OFt-el*EbSp*-)o+tR$wF!LrW`)Ae#M<{T'
            '&9BdFSQl7b*a$K1!&AA^Hy(I_@ucrscc1r_vKW<pH@91jx0q4Sw;1`0dcF~V!igKu1h5+3k|k`FHTi6C&s)9B%sSyLLs}B}HHWpG'
            '?FPF=c{C>VsOlt>cI^fu#OreK1GM56#x609+1B~3t7=w_2EM{K>HfhG8`JgC;mfX)qpAm5S(2>J7V$HTPMqBJG#$3iIaLaND90(('
            'YW_`N2%RMl%Om_QCL8u5TlMlR?kOdtjEE^pFq+GWoe2LY$0K~*BhJ3wVebdGhzB|IaJS4hu+Zv0r3gbTrgc+_EGPMOBuGz7N+oQj'
            'Yr1N)p1NrZ24ia~H(U9DgU)P_#4+WO5aT|^9zBcNRvl2CH$QhyPEOB%>7AUu>6~AkUYx__{U0xX^ptpWand;pm?af9Pa{#yX?!2<'
            'U_`$AJvqqdi=dBpJOzFCj~O)ypM%jN(E&<q8RGOeJH<Q6kO@~F#%^9|vgc6cTk2)w*?1bIoHQVmR4*zKT=8S@_khJp#hx&n=Gh8)'
            'jYh?D#dl6|hCSl!#=<&azK)p2MAQCzpi20jCbEs{N9kvJZsX#I^{{g8p-d;Y5px<PGSw2YZ-llbdV!lCp`qxtVz<<m6|y}a4Q^+='
            '$w>$+Wtw@GPpjy7a5s)<wNRrPMDK9^E>M{zI96PVjSa=gF48$H>Y|sK<>ZR=q!D)<BKev54Sh=WX3%$6c3*rNt3^X+UC*qGPrZr6'
            'ao7VeA35PWEPg`@;aq}Xki^FtI_vmd=v3M(D&>h+z&(nS1$Bi+Jh*z@Zop+EBvjEymWFl4ESjL0p}Q3I&@W9Hc~MFUUMM014bZgm'
            'x}J1anPki(DUf^+^sSw23v1cpcPS?N_jA9D7PB1W4p68uJ?orE8ult3ELA>}?G6{CAKm(%*U#cs{0P5`h1IM$Bk3|Kp9(}+svbw*'
            '8O9D>Yb!jOs&H=VoJAG+-#MYzEPJxU!OI3s1M+nECAHL#yi^*R-Vu=`j)7yKroXIP2*qa@gpH%8Rl)pZOtu4LEPi`}x!4-4Kquk#'
            '8PV@RfL^3o!B(XlJB+l4NvvuCM~fAmq6eTi{JozhhJSmaH$XQ48asi;<|X^L5l6)J*U^w{&*IZw>Ajqc1gC?=8jizzUHwL~oCfzs'
            'BQ|dEn%!m@dgZye=8(ZV-a-`2#x-^8<!~u^lKM#`8nV!z;8NIeyLWF@wFyS!G#&?VI;wujTQ@Qfyqv-cN0Le@$qzl(6R|yoSC-Fe'
            'ey{9`M^Un6OP3zy&#-QgC%H8&n3`G|WazZlqE)bb&QkgEmsL+tVgA!9!b5?2rvy}ATN!bx&S|#_ioDS#dn#rmxwgeL$KXp}^ZV(^'
            'O4BYVddMzSh?zAEQzpTV_Qa7X0YuuPhJRqQBE1g=+cK?%&TE2b4GMhz8MEt7RuLVPi|H(a!%~*`G)V*6^{JdU@q*$8VcmTSz6;hg'
            'il6oZIw(Jnm)vkJiZWU+GD?5q(nOL}hvfq|@()GHfs|b9da2TZRg%Vlt8E1bJQDmYd~|M5xbdxAn9%lp6mY46@8nTMFEwA>B~ztL'
            '7t~=r-oSg>!rRk_k3Bsg=N$$O1Zv29nz~aoicr+kl_2;tA4~}WxL@RGlp9N=?Zf5)x|bF!vieAE%t^}ZG?eBFLzw3rCmX~$NWmyP'
            'H-nP|wkAs4$W=|%jmCFEp-MSq)amQXgkA2dGqQ&=vIEY6ilim(B6i1+7;1Ry(VucU+2d+$6N>pJPL$>WQH#_L@edrD2EDxw(&0-r'
            'hnNRmClc=I(a5$;Co;7o9&0fL#gC>F`g73y)Aw$$^NFetW4y)D7Y)-zhA~*k2tt-6MY^k^?bM6hgyon4V}Lpdx2fLLe7D5n$u>FS'
            'M@oZe0Lzmr)a~t~yoK+Tl|iAB_I#>JEG|y#t>zRI(@dKHQ7ey_RB63MAwZA+)H24_mRPx2nyWRWCJ{`8nig6%30?$LC(d=5Pc?;|'
            'Q<EBz<#gSW<O8##pfS5P>*}h7B6f=2U~Tq|-abWhDlf%~wejN*?%MbzO_CU-zu4S%!sgG6<%Vgb#v!2%1o#^7XdvNUTx4yKClskF'
            '>iFFUX0qIaV%RCpAXZic8^XlzArO;#Wm&j%7lN98dE`1Aq5!-Y$CN;$a_WtaU57IiHd44E)k>iDyA*>$=s|9C=h=F7{Y8^-+`O_>'
            'ZHR0X^nylkh!8mtK==8LG&sVmfR~%6UU&Jcf4a^$dI~H?6ipNd3ihO|{qQ5Dh<ur&V)v+_0JWF~5HX0liK1OgGwscDZ9Y3R#&pAd'
            'H<v_LwvIkkd+_!MZp@n8oU%n2t0=#j+vX2Hc&^O1Tu#|<0tPc<&V&Y>O1R|ll;!gTn>0_Nw4s}zFHE&%q$zSG*>?u37gC~L_uMlz'
            'zE)HbSQtiE<eFhKavLs~Q5xOh-7a?ISM?(b4o}B~35RWY@#>XnBCAf2h8D`Gs8N&q_ssg7><`c^Of&fSMAXhd*t^vN>K3hm?Us+u'
            'oBdVL_GTBYVb%F3m~~CA(-TRqXanpo-=Fu|=kHE`xorP-`tC#TvUAmGU;g#A8kIh$^Pr#&Br=2nSMlbIVb@{ptP^pvuUv<qi>zjS'
            '$8IqjfG+MQvl!);j!+D+^A`8Dg_05IlB;6HIZ}ft+bP&9QFm<!ww}EEQxNdlDD>ezfcpF8nd}FHJRT)zt)zNztlRG{E<Xs{ncx?6'
            'Bogc+3R^+JBaYU)nIPa~)K6ELf$bMWMc<sdJ{+75Xu~U8s%7igFl~>zGHCktMYU;Dxt=_^C=08n+3wI=sxWre!m72!7#Iw{O0#GU'
            'x8l5x<UjLdogw>r8X<sb76lJw;SNE^7O)dS*Y|?k@o+Fs8Pr}t+M8us`7Z;sdxy1*aQ`Yn{b6}+LZR|kp46X|<e*e6AR%W}D}PU{'
            'J_nCZGXALJmhC>#a6AccHk_sm^3^cOoJ`}N7b$Afah_tPv<$=u5cLQ7I-l^43Rd|9^B&{OX3?bXq!^cd+wyl*Cq~_xg7T-9RP#^V'
            'WP;9lXnWm+%4-?j5}bfX33n6>Zn^)hd+yif;aw&312sn3c)duE|E)I^4~w~tzJ|}?*{lzgkjU?Ja1v(naZo$lQ%k?GeRX>r5??J$'
            'U%lcHc2Wb^;NLuCI^8jx1=@uA=)9MXjn&E<ROCyt7DG2syWfTuYe_;4zep1FK%o@+VCk(YSh@*&wQki%fL}|8@=52s^Wmy<7DT2D'
            '7=h@QAG>(r_lVM)p$G*tjo4z97kjUKAQM6uj^ljBIh`0-JWD?7i#AKva4M|FbEMz<>>S9|SX^`BH9XVA<CX;A$3r`9SZCz|+k*47'
            'uCi!^KK#%@ZYp%A3){pr2@0WEAuicq%Lf6AHvaJ1!F^hw^P*T;Y%Z<VqM&yln5}XOlw0)J9rF998h@20Ndt4zGkYwfp@LB!rLjsf'
            '{-^p^Pt{Gy8xigB2Jxfsm!K}kKO-1AiEEen9O=kOnvz=s!SJ#j5Eh@VdIk2BG6TY6^#9^}R#nkRrqf$$u+1b|&!Y(^nGI><6dAfe'
            'gSbp?aw2hFAZcMt2^WTo<vJzSwm6{r@mGrOC4P4tKaj<cAR51f$5Yn$L~BA;$>oYds71{=Lx$(z)hnWT-w$pgG<-BZ-i|XQT-y)c'
            'uCFjx?g?3c;q!DnNEaE#{A1pyg07Ai);yZ7GeV2N{yS#0NOdCKJ?48~MYAR1K=3{Lgv{H~_E=$m)R|Ir7O~L+d(L=Fug+DThmxf1'
            'pYt8tJGw3BFY7`_!M!oFn;;*;Rsjh(m_(}?3`hBB{_y_Sv(7mt#G{-Iy0#vus_h<V!<bs<1^o6QpJ1f@B*3#9WVsBJQ$_1E8aeVe'
            '3hcuO<aWewkCHT*Ovk}(G?{`z22$nico{6FaX!M!V3n*v6;4*xkuMe*y_DhJosZ|E8ZAj!t4HP;U`PS)KeR7TkG5t;CR9)}cpc4n'
            'neXF*78I=>D<f!yhiyE^2}bnpm_RoV4c5;&s4h?##)comfir3)y=Lpk#OP0gR>5Z-5%a%-R9E}'
        ),
    },
}

def _load_local_pipeline_modules():
    # 1) Zaten import edilebiliyorsa (yerel .venv, repo Colab'da klonlanmis) dogrudan kullan.
    try:
        import local_pipeline
        modules = {}
        for _name in _LOCAL_PIPELINE_MODULE_NAMES:
            modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
        print('local_pipeline modulleri dogrudan import edildi:', list(modules))
        return modules
    except Exception:
        pass

    # 2) Drive'da veya proje kokunun bir ustunde daha once birakilmis bir kopya var mi?
    search_roots = []
    try:
        search_roots.append(Path(DRIVE_PROJECT_ROOT))
    except NameError:
        pass
    search_roots += [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    for root in search_roots:
        candidate_dir = Path(root) / 'local_pipeline'
        if all((candidate_dir / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(candidate_dir.parent))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri onceden birakilmis kopyadan yuklendi:', candidate_dir)
                return modules
            except Exception:
                pass

    # 3) PROJECT_REPO_URL verilmisse git clone dene.
    if PROJECT_REPO_URL:
        clone_dir = Path('/content/youtube_automation_repo') if os.path.isdir('/content') else Path.cwd() / '_youtube_automation_repo'
        if not (clone_dir / 'local_pipeline' / (_LOCAL_PIPELINE_MODULE_NAMES[0] + '.py')).is_file():
            subprocess.run(['git', 'clone', '--depth', '1', PROJECT_REPO_URL, str(clone_dir)], check=False)
        if all((clone_dir / 'local_pipeline' / f'{_n}.py').is_file() for _n in _LOCAL_PIPELINE_MODULE_NAMES):
            sys.path.insert(0, str(clone_dir))
            try:
                modules = {}
                for _name in _LOCAL_PIPELINE_MODULE_NAMES:
                    modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
                print('local_pipeline modulleri PROJECT_REPO_URL klonundan yuklendi:', clone_dir)
                return modules
            except Exception:
                pass

    # 4) Self-contained fallback: bu hucreye gomulu, build sirasinda
    # local_pipeline/<modul>.py'den birebir uretilmis base64 kopyalari, SHA-256
    # hash'lerini dogrulayarak diske yazip import et.
    fallback_dir = Path(tempfile.mkdtemp(prefix='youtube_automation_bootstrap_'))
    pkg_dir = fallback_dir / 'local_pipeline'
    pkg_dir.mkdir(parents=True, exist_ok=True)
    (pkg_dir / '__init__.py').write_text('', encoding='utf-8')
    for _name, _entry in _LOCAL_PIPELINE_PAYLOADS.items():
        if _entry.get('encoding') != 'zlib+b85':
            raise RuntimeError(f"Desteklenmeyen gomulu payload formati: {_entry.get('encoding')!r}")
        _source_bytes = zlib.decompress(base64.b85decode(_entry['data']))
        _actual_digest = hashlib.sha256(_source_bytes).hexdigest()
        if _actual_digest != _entry['sha256']:
            raise RuntimeError(
                f"Gomulu local_pipeline.{_name} payloadi bozuk/tutarsiz: "
                f"beklenen sha256={_entry['sha256']!r}, hesaplanan={_actual_digest!r}. "
                "Bu notebook yeniden uretilmeli (nb_builder ile)."
            )
        (pkg_dir / f'{_name}.py').write_text(_source_bytes.decode('utf-8'), encoding='utf-8')
    sys.path.insert(0, str(fallback_dir))
    modules = {}
    for _name in _LOCAL_PIPELINE_MODULE_NAMES:
        modules[_name] = __import__(f'local_pipeline.{_name}', fromlist=[_name])
    print('local_pipeline modulleri: repo/Drive/PROJECT_REPO_URL bulunamadi; gomulu (self-contained, hash dogrulanmis) kopyalar kullanildi.')
    return modules

_local_pipeline_modules = _load_local_pipeline_modules()
youtube_data = _local_pipeline_modules['youtube_data']
viral_prediction = _local_pipeline_modules['viral_prediction']
google_ai = _local_pipeline_modules['google_ai']



In [ ]:
# Notebook 04 ile ayni kalici yol cozumleme (Faz 9): Drive (Colab) / /content (Colab, Drive kapali) / yerel results/
youtube_paths = youtube_data.resolve_project_paths(
    is_colab=IS_COLAB,
    use_google_drive=USE_GOOGLE_DRIVE,
    project_root=PROJECT_ROOT,
    drive_project_root=DRIVE_PROJECT_ROOT,
)
print("YouTube veri arama dizini (Notebook 04 ciktisi bekleniyor):", youtube_paths["youtube_data_dir"])


## 1. Gerçek geçmiş veri setini yükle (Notebook 04 çıktısı)

Notebook 04'ün yazdığı `normalized_videos.csv` + `collection_manifest.json` dosyaları önce Drive-kalıcı
konumda, sonra Colab geçici `/content/results/...` konumunda, sonra yerel proje `results/...` konumunda
aranır (Faz 9). `collection_manifest.json` okunur ve toplama durumu `failed`/`insufficient_data` ise bu
notebook **kendinden emin bir tahmin üretmeyi reddeder** — sadece ham geçmiş satır sayısına değil, Notebook
04'ün kendi topladığı verinin güvenilirliğine de bakılır.

**Önemli sınır:** Burada kullanılan `view_count`/`like_count`/`comment_count` **herkese açık** YouTube Data
API v3 istatistikleridir. Bunlar YouTube Analytics'in **özel** metrikleriyle (CTR, izlenme süresi/retention,
gelir) **aynı şey değildir** — bu notebook hiçbir zaman CTR/retention/gelir tahmini yapmaz, sadece
görüntülenme/beğeni/yorum sayıları üzerinden basit istatistiksel bir taban çizgisi kurar.

In [ ]:
def find_first_existing(candidates):
    for candidate in candidates:
        if candidate and Path(candidate).is_file():
            return Path(candidate).resolve()
    return None

NORMALIZED_VIDEOS_PATH = ''  # elle yol vermek istersen buraya yaz

# Faz 9: Drive-kalici -> Colab gecici /content -> yerel results/ sirasiyla ara.
youtube_data_search_dirs = [
    str(youtube_paths["youtube_data_dir"]),
    str(RESULTS_DIR / 'youtube_data'),
    'results/youtube_data',
    '../results/youtube_data',
    '/content/results/youtube_data',
]
normalized_path = None
if NORMALIZED_VIDEOS_PATH and Path(NORMALIZED_VIDEOS_PATH).is_file():
    normalized_path = Path(NORMALIZED_VIDEOS_PATH).resolve()
else:
    found = youtube_data.find_normalized_videos_csv(youtube_data_search_dirs)
    if found is not None:
        normalized_path = found

# collection_manifest.json: Notebook 04'un TOPLAMA SURECININ KENDISI hakkindaki
# beyani. Satir sayisi yeterli olsa bile, manifest status'u 'failed' veya
# 'insufficient_data' ise bu notebook kendinden emin bir tahmin URETMEZ (asagidaki
# veri yeterliligi kapisinde kontrol edilir).
collection_manifest = None
if normalized_path is not None:
    manifest_path = normalized_path.parent / 'collection_manifest.json'
    if manifest_path.is_file():
        try:
            with open(manifest_path, 'r', encoding='utf-8') as f:
                collection_manifest = json.load(f)
        except Exception as exc:
            print(f'UYARI: collection_manifest.json okunamadi: {exc}')

collection_status = collection_manifest['status'] if collection_manifest else None
print('Notebook 04 toplama durumu (collection_manifest.json):', collection_status or '(manifest bulunamadi)')

# Faz 9: onbellek/veri bayatligi uyarisi -- toplama ne kadar eski?
YOUTUBE_DATA_STALENESS_WARNING_HOURS = 48
if collection_manifest and collection_manifest.get('completed_at'):
    try:
        completed_at = datetime.fromisoformat(collection_manifest['completed_at'])
        if completed_at.tzinfo is None:
            completed_at = completed_at.replace(tzinfo=timezone.utc)
        age_hours = (datetime.now(timezone.utc) - completed_at).total_seconds() / 3600.0
        if age_hours > YOUTUBE_DATA_STALENESS_WARNING_HOURS:
            print(f'UYARI: YouTube verisi {age_hours:.1f} saat once toplanmis '
                  f'(esik: {YOUTUBE_DATA_STALENESS_WARNING_HOURS}s). Guncel sonuclar icin Notebook 04\'u tekrar calistir.')
    except Exception:
        pass

# Faz 6 duzeltmesi: Notebook 04 artik normalized_videos.csv'yi HER ZAMAN sabit
# semali yazar, ama bu notebook yine de kendini savunmali okumali - dosya elle
# silinmis/bozulmus/baska bir surumden kalma olabilir. Hicbir durumda cokmez:
# headerless/tamamen bos CSV -> EmptyDataError yakalanir; gerekli sutunlar eksikse
# ayri bir kontrolle yakalanir. Her iki durumda da history_df sabit semali BOS
# bir DataFrame'e duser ve asagidaki veri yeterliligi kapisi bunu insufficient_data
# olarak raporlar.
EMPTY_HISTORY_COLUMNS = [
    'video_id', 'title', 'channel_id', 'channel_title', 'published_at',
    'view_count', 'like_count', 'comment_count', 'duration_iso8601',
]
REQUIRED_HISTORY_COLUMNS = ['video_id', 'channel_id', 'view_count', 'published_at']

history_df = pd.DataFrame(columns=EMPTY_HISTORY_COLUMNS)
history_load_status = 'insufficient_data'
history_load_reason = 'normalized_videos.csv bulunamadi'

if normalized_path is None:
    print(f'UYARI: {history_load_reason}. Once Notebook 04\'u YOUTUBE_DATA_API_KEY ve '
          'gercek SEED_CHANNEL_IDS ile calistir. history_df bos kalacak.')
else:
    raw_history_df = None
    try:
        raw_history_df = pd.read_csv(normalized_path)
    except pd.errors.EmptyDataError:
        history_load_reason = f'{normalized_path} tamamen bos (basliksiz/0 byte) bir CSV dosyasi'
        print(f'UYARI: {history_load_reason}. history_df bos kalacak, tahmin uretilmeyecek.')
    except Exception as exc:
        history_load_reason = f'{normalized_path} okunurken hata: {exc}'
        print(f'UYARI: {history_load_reason}. history_df bos kalacak.')

    if raw_history_df is not None:
        missing_columns = [c for c in REQUIRED_HISTORY_COLUMNS if c not in raw_history_df.columns]
        if missing_columns:
            history_load_reason = (
                f'gerekli sutunlar eksik: {missing_columns} '
                f'(dosyada bulunanlar: {list(raw_history_df.columns)}). Dosya bozuk veya eski/uyumsuz '
                'bir Notebook 04 semasindan kalma olabilir.'
            )
            print(f'UYARI: {history_load_reason}. history_df bos kalacak.')
        else:
            candidate_df = raw_history_df.dropna(subset=['view_count']).copy()
            if 'published_at' in candidate_df.columns:
                candidate_df['published_at'] = pd.to_datetime(candidate_df['published_at'], errors='coerce', utc=True)
                candidate_df = candidate_df.sort_values('published_at')
            if len(candidate_df) == 0:
                history_load_reason = (
                    f'{normalized_path} gecerli semaya sahip ama kullanilabilir (view_count dolu) satir yok'
                )
                print(f'UYARI: {history_load_reason}. history_df bos kalacak.')
            else:
                history_df = candidate_df
                history_load_status = 'ok'
                history_load_reason = ''
                print('Yuklendi:', normalized_path, f'({len(history_df)} kullanilabilir satir, view_count dolu)')

## 2. Manuel örnekler — SADECE deneysel fallback

Bu örnekler asla gerçek geçmiş veri setine karıştırılmaz ve asla `production_ready`/güvenilir tahmin üretmek için kullanılmaz — sadece heuristik formülün mantığını göstermek için elle doldurulabilir.

In [ ]:
MANUAL_EXAMPLES = [
    {
        'title': 'Bu Sahneyi Kimse Beklemiyordu!',
        'duration_sec': 58,
        'channel_avg_views_last_10': 42000,
        'channel_avg_ctr': 0.061,
    },
]

def heuristic_predict(v):
    base = v['channel_avg_views_last_10']
    duration_penalty = 1.0 if 30 <= v['duration_sec'] <= 90 else 0.85
    ctr_factor = v['channel_avg_ctr'] / 0.05
    return base * duration_penalty * ctr_factor

manual_preds = [round(heuristic_predict(v)) for v in MANUAL_EXAMPLES]
log_result(
    'viral-prediction', 'manual-example-heuristic', None, 0.0, 0.0,
    f'SADECE gosterim amacli, {len(MANUAL_EXAMPLES)} elle girilen ornek: {manual_preds}',
    status='experimental',
)
print('Manuel ornek tahminleri (SADECE gosterim, production icin kullanilmaz):', manual_preds)

## 3. Veri yeterliliği kapısı

In [ ]:
MIN_HISTORY_ROWS = 20  # bu esigin altinda notebook kendinden emin tahmin URETMEZ

HISTORY_ROW_COUNT = len(history_df)
COLLECTION_STATUS_OK = collection_status not in ('failed', 'insufficient_data')
DATA_SUFFICIENT = (HISTORY_ROW_COUNT >= MIN_HISTORY_ROWS) and COLLECTION_STATUS_OK

date_range = {'start': None, 'end': None}
if HISTORY_ROW_COUNT and 'published_at' in history_df.columns and history_df['published_at'].notna().any():
    date_range['start'] = history_df['published_at'].min().isoformat()
    date_range['end'] = history_df['published_at'].max().isoformat()

print(f'Gecmis satir sayisi: {HISTORY_ROW_COUNT} (esik: {MIN_HISTORY_ROWS})')
print('Veri yukleme durumu:', history_load_status, '-', history_load_reason or '(sorun yok)')
print('Notebook 04 toplama durumu:', collection_status or '(manifest bulunamadi)')
print('Veri tarih araligi:', date_range)
if not COLLECTION_STATUS_OK:
    print(f"UYARI: collection_manifest.json status='{collection_status}' -- Notebook 04'un toplama sureci "
          "basarisiz/yetersizdi. Satir sayisi yeterli olsa bile kendinden emin tahmin URETILMEYECEK.")
elif not DATA_SUFFICIENT:
    print(f'UYARI: Yetersiz veri (status=insufficient_data olacak). En az {MIN_HISTORY_ROWS} satir icin '
          "Notebook 04'u daha fazla YOUTUBE_CHANNEL_INPUTS ile calistir.")
else:
    print('Veri yeterli; baseline degerlendirmesi calisacak.')


## 4. Şeffaf baseline yöntemler

Üç açıklanabilir, ücretsiz, API gerektirmeyen baseline: kanal medyan izlenme, son N videonun hareketli medyan/ortalaması, etkileşim (like/comment oranı) ile ayarlanmış medyan.

In [ ]:
# Kanonik baseline yontemleri (local_pipeline/viral_prediction.py). Master
# notebook da AYNI fonksiyonlari cagirir -- iki ayri kopya yoktur.
BASELINE_METHODS = viral_prediction.BASELINE_METHODS
ROLLING_WINDOW = viral_prediction.DEFAULT_ROLLING_WINDOW
zero_safe_smape = viral_prediction.zero_safe_smape
print(f'{len(BASELINE_METHODS)} kanonik baseline yontemi yuklendi:', list(BASELINE_METHODS))


## 5. Zaman-tabanlı holdout değerlendirmesi

Veri yayın tarihine göre sıralanır; son `%HOLDOUT_FRACTION` bölüm test (holdout), geri kalanı eğitim/referans olarak kullanılır — bu, gerçek üretim senaryosuna (geçmişten geleceği tahmin) daha yakın bir değerlendirmedir.

In [ ]:
HOLDOUT_FRACTION = 0.2
MIN_HOLDOUT_ROWS = 5

baseline_metrics = {}
train_df, holdout_df = pd.DataFrame(), pd.DataFrame()

if DATA_SUFFICIENT:
    t0 = time.time()
    # Kanonik zaman-tabanli holdout degerlendirmesi (master notebook ile ayni).
    evaluation = viral_prediction.evaluate_baselines_on_holdout(
        history_df, holdout_fraction=HOLDOUT_FRACTION, min_holdout_rows=MIN_HOLDOUT_ROWS,
    )
    train_df = evaluation['train_df']
    holdout_df = evaluation['holdout_df']
    baseline_metrics = evaluation['metrics']
    latency = time.time() - t0

    if len(holdout_df) < MIN_HOLDOUT_ROWS:
        print(f'UYARI: holdout seti cok kucuk ({len(holdout_df)} satir); metrikler guvenilmez olabilir.')

    for method_name in BASELINE_METHODS:
        if method_name in baseline_metrics:
            m = baseline_metrics[method_name]
            log_result('viral-prediction', method_name, None, 0.0, latency / max(len(BASELINE_METHODS), 1),
                       f"MAE={m['mae']:.0f} medianAE={m['median_ae']:.0f} sMAPE={m['smape_pct']}% n={m['sample_count']}",
                       status='review_required')
        else:
            log_result('viral-prediction', method_name, None, 0.0, 0.0,
                       'holdout icin tahmin uretilemedi', status='insufficient_data')
    print(json.dumps(baseline_metrics, indent=2))
else:
    log_result('viral-prediction', 'baseline-evaluation', None, 0.0, 0.0,
                f'yetersiz veri ({HISTORY_ROW_COUNT} < {MIN_HISTORY_ROWS})', status='insufficient_data')
    print('Baseline degerlendirmesi atlandi: yetersiz veri.')


## 6. Opsiyonel LLM tahmini (istatistiksel baseline ile karşılaştırılır)

**Kalite politikası:** LLM tahmini SADECE `DATA_SUFFICIENT=True` iken çalışır ve SADECE gerçek geçmiş holdout verisiyle beslenir — asla `MANUAL_EXAMPLES`'tan güven uydurmaz. Küçük bir holdout örneklemiyle sınırlıdır (maliyet + anlamlılık için).

In [ ]:
LLM_HOLDOUT_SAMPLE_SIZE = 5
llm_metrics = None
llm_provider_used = None

# Faz 8 duzeltmesi: OpenAI response_format={'type':'json_object'} bir JSON NESNESI
# ister, dizi degil. Onceki surum bunu isterken promptta hala bir JSON DIZISI
# istiyordu - tutarsizdi. Simdi TUM saglayicilar icin ayni sema istenir: ust
# seviyede {"predictions": [...]}. Her tahmin, listedeki POZISYONA gore degil,
# video_id (sabit/kararli bir tanimlayici) ile eslestirilir.
def parse_prediction_response(raw):
    try:
        start = raw.find('{'); end = raw.rfind('}') + 1
        if start == -1 or end <= start:
            return None
        obj = json.loads(raw[start:end])
        if not isinstance(obj, dict) or not isinstance(obj.get('predictions'), list):
            return None
        return obj['predictions']
    except Exception:
        return None

if not DATA_SUFFICIENT:
    print('LLM tahmini atlandi: yetersiz gercek veri (birkac elle girilen ornekten guven uydurulmaz).')
elif not (OPENAI_API_KEY or ANTHROPIC_API_KEY or GOOGLE_API_KEY):
    print('LLM tahmini atlandi: API anahtari yok.')
else:
    llm_sample = holdout_df.head(LLM_HOLDOUT_SAMPLE_SIZE)
    channel_context = (
        train_df.groupby('channel_id')['view_count']
        .median().round(0).to_dict()
    )
    prompt_payload = [
        {
            'video_id': str(row['video_id']),
            'title': row['title'],
            'channel_id': row['channel_id'],
            'channel_median_views': channel_context.get(row['channel_id']),
            'published_at': str(row['published_at']),
        }
        for _, row in llm_sample.iterrows()
    ]
    PREDICTION_PROMPT = (
        "Sen bir YouTube analitik uzmanisin. Asagidaki gercek videolarin her biri icin, verilen kanal "
        "medyan izlenme sayisina gore beklenen izlenme sayisini (predicted_views, tek sayi) ve 0-1 arasi "
        "guven skoru (confidence) ver. Her tahmin icin verilen video_id degerini AYNEN geri don - "
        "eslestirme icin kullanilacak, liste sirasina guvenilmeyecek. TUM videolar icin tahmin uret, "
        "hicbirini atlama. SADECE su sema ile bir JSON NESNESI don (dizi DEGIL): "
        "{{\"predictions\": [{{\"video_id\": \"...\", \"predicted_views\": <int>, \"confidence\": <0-1 sayi>}}]}}\n\n"
        "Veri:\n{data}"
    ).format(data=json.dumps(prompt_payload, ensure_ascii=False))

    predictions, provider_name, cost, latency = None, None, 0.0, 0.0
    try:
        if OPENAI_API_KEY:
            from openai import OpenAI
            client = OpenAI(api_key=OPENAI_API_KEY)
            t0 = time.time()
            resp = client.chat.completions.create(
                model='gpt-5.6-luna', messages=[{'role': 'user', 'content': PREDICTION_PROMPT}],
                temperature=0, response_format={'type': 'json_object'},
            )
            latency = time.time() - t0
            raw_text = resp.choices[0].message.content
            predictions = parse_prediction_response(raw_text)
            provider_name = 'openai-gpt-5.6-luna'
            usage = resp.usage
            cost = (usage.prompt_tokens / 1_000_000) * 1.00 + (usage.completion_tokens / 1_000_000) * 6.00
        elif ANTHROPIC_API_KEY:
            import anthropic
            client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
            t0 = time.time()
            resp = client.messages.create(
                model='claude-haiku-4-5', max_tokens=600,
                messages=[{'role': 'user', 'content': PREDICTION_PROMPT}],
            )
            latency = time.time() - t0
            predictions = parse_prediction_response(resp.content[0].text)
            provider_name = 'anthropic-claude-haiku-4-5'
            cost = (resp.usage.input_tokens / 1_000_000) * 1.00 + (resp.usage.output_tokens / 1_000_000) * 5.00
        elif GOOGLE_API_KEY:
            # Google: SABIT MODEL YOK -- kanonik router (local_pipeline/google_ai.py).
            # Viral sonucun yorumlanmasi KRITIK OLMAYAN bir istir: Rota B
            # (gemini-3.5-flash-lite -> gemini-3.1-flash-lite -> gemma-4-26b-a4b-it).
            # Istatistiksel baseline'in YERINE GECMEZ; onunla karsilastirilir.
            _g_credential = google_ai.resolve_secret('google_api', get_secret)
            _g_client = google_ai.build_google_client(_g_credential)
            _g_manifest = google_ai.build_google_capability_manifest(
                _g_client, credential=_g_credential)
            _g_ledger = google_ai.QuotaLedger()
            print(google_ai.format_route_line(
                google_ai.select_google_route('viral_interpretation',
                                              manifest=_g_manifest, ledger=_g_ledger),
                _g_credential))
            _g_result = google_ai.run_google_viral_interpretation(
                PREDICTION_PROMPT, client=_g_client, manifest=_g_manifest, ledger=_g_ledger)
            latency = _g_result.get('latency_sec', 0.0)
            provider_name = 'google-' + str(_g_result.get('model') or 'router')
            cost = 0.0  # ucretsiz katman; gercek butce RPM/TPM/RPD kotasidir
            if _g_result['status'] == 'ok':
                predictions = parse_prediction_response(_g_result['output'])
            else:
                # Cikti UYDURULMAZ: predictions None kalir -> asagida 'failed' loglanir.
                predictions = None
                print('Google rotasi basarisiz:', _g_result['status_reason'])

        if predictions is None:
            log_result('viral-prediction', provider_name or 'llm-unknown', None, cost, latency,
                        "LLM cevabi ayristirilamadi (beklenen sema: {'predictions': [...]})", status='failed')
        else:
            # video_id ile eslestir (liste POZISYONUNA guvenme - Faz 8 duzeltmesi).
            # Gercek holdout view_count degerleri (actuals) HICBIR sekilde LLM
            # ciktisindan degil, dogrudan llm_sample'dan alinir.
            llm_sample_by_id = {str(row['video_id']): row for _, row in llm_sample.iterrows()}
            matched_ids, matched_preds, matched_actuals = [], [], []
            unknown_ids, malformed_count = [], 0
            for entry in predictions:
                if not isinstance(entry, dict) or 'video_id' not in entry or 'predicted_views' not in entry:
                    malformed_count += 1
                    continue
                vid = str(entry['video_id'])
                if vid not in llm_sample_by_id:
                    unknown_ids.append(vid)
                    continue
                if vid in matched_ids:
                    continue  # duplicate video_id in response - ignore repeats
                matched_ids.append(vid)
                matched_preds.append(entry['predicted_views'])
                matched_actuals.append(float(llm_sample_by_id[vid]['view_count']))

            expected_count = len(llm_sample)
            matched_count = len(matched_ids)
            is_complete = (matched_count == expected_count) and malformed_count == 0 and not unknown_ids
            completeness_note = (
                'tam' if is_complete else
                f'EKSIK/BOZUK ({matched_count}/{expected_count} eslesti, malformed={malformed_count}, bilinmeyen_video_id={len(unknown_ids)})'
            )

            if not matched_preds:
                log_result('viral-prediction', provider_name, None, cost, latency,
                            f'hic gecerli eslesen tahmin yok ({completeness_note})', status='failed')
            elif not is_complete:
                # Faz 8 duzeltmesi: eksik/bozuk cikti acikca 'failed' olarak isaretlenir,
                # kismi sonuc sessizce baseline ile karsilastirmaya sokulmaz.
                log_result('viral-prediction', provider_name, None, cost, latency,
                            f'eksik/bozuk LLM ciktisi, sonuc guvenilir sayilmadi ({completeness_note})', status='failed')
            else:
                mae = float(np.mean(np.abs(np.array(matched_preds) - np.array(matched_actuals))))
                median_ae = float(np.median(np.abs(np.array(matched_preds) - np.array(matched_actuals))))
                smape = zero_safe_smape(matched_actuals, matched_preds)
                llm_metrics = {
                    'mae': round(mae, 1), 'median_ae': round(median_ae, 1),
                    'smape_pct': round(smape, 2) if smape is not None else None,
                    'sample_count': matched_count, 'expected_count': expected_count, 'complete': True,
                }
                llm_provider_used = provider_name
                log_result('viral-prediction', provider_name, None, cost, latency,
                            f"MAE={mae:.0f} medianAE={median_ae:.0f} n={matched_count} ({completeness_note}, baseline ile karsilastir)",
                            status='review_required')
    except Exception as exc:
        log_result('viral-prediction', provider_name or 'llm-unknown', None, cost, latency, f'API hatasi: {exc}', status='failed')

    if llm_metrics and baseline_metrics:
        best_baseline_mae = min(m['mae'] for m in baseline_metrics.values())
        llm_mae = llm_metrics['mae']
        comparison_word = 'daha iyi' if llm_mae < best_baseline_mae else 'daha kotu ya da esit'
        print(f'LLM MAE={llm_mae} vs en iyi baseline MAE={best_baseline_mae} ({llm_provider_used} {comparison_word})')

## 7. Dışa aktarım

Tek bir "kesin" sayı yerine **aralık** (`predicted_range`) ve açık güven seviyesi/sınırlamalar raporlanır.

In [ ]:
df = pd.DataFrame(results)
out_path = RESULTS_DIR / '05_viral_prediction_sonuclar.csv'
if not df.empty:
    display(df)
    df.to_csv(out_path, index=False)
else:
    pd.DataFrame(columns=['task', 'candidate', 'quality_score_1_5', 'cost_usd_estimate', 'latency_sec', 'notes', 'status']).to_csv(out_path, index=False)
print('Kaydedildi:', out_path)

limitations = []
if history_load_status != 'ok':
    limitations.append(f'gecmis veri yuklenemedi/yetersiz: {history_load_reason}')
if not DATA_SUFFICIENT:
    limitations.append(f'gercek gecmis veri yetersiz ({HISTORY_ROW_COUNT} < {MIN_HISTORY_ROWS} satir)')
if HISTORY_ROW_COUNT and len(holdout_df) < MIN_HOLDOUT_ROWS:
    limitations.append(f'holdout seti kucuk ({len(holdout_df)} satir); metrikler gurultulu olabilir')
if llm_metrics is None:
    limitations.append('LLM tahmini calismadi, atlandi veya eksik/bozuk cikti verdi; sadece istatistiksel baseline mevcut')
limitations.append('YouTube Shorts algoritmasindaki degisiklikler, mevsimsellik ve konu turu farkliliklari modele dahil edilmemistir')

if not DATA_SUFFICIENT:
    prediction_status = 'insufficient_data'
    status_reason = f'gercek gecmis veri {MIN_HISTORY_ROWS} satir esiginin altinda ({HISTORY_ROW_COUNT} satir) - {history_load_reason or "yeterli veri toplanamadi"}'
    predicted_range = None
    confidence_level = 'none'
    chosen_method = None
elif baseline_metrics:
    chosen_method = min(baseline_metrics, key=lambda name: baseline_metrics[name]['mae'])
    chosen = baseline_metrics[chosen_method]
    center = float(train_df['view_count'].median())
    predicted_range = {'low': round(max(center - chosen['mae'], 0), 0), 'high': round(center + chosen['mae'], 0)}
    if HISTORY_ROW_COUNT >= 200:
        confidence_level = 'medium'
    elif HISTORY_ROW_COUNT >= 50:
        confidence_level = 'low'
    else:
        confidence_level = 'very_low'
    prediction_status = 'review_required'
    status_reason = f'istatistiksel baseline ({chosen_method}) holdout uzerinde degerlendirildi; insan onayi bekleniyor'
else:
    chosen_method = None
    predicted_range = None
    confidence_level = 'none'
    prediction_status = 'failed'
    status_reason = 'veri yeterliydi ama hicbir baseline holdout tahmini uretemedi'

viral_prediction = {
    'status': prediction_status,
    'status_reason': status_reason,
    'dataset_size': HISTORY_ROW_COUNT,
    'date_range': date_range,
    'min_history_rows_threshold': MIN_HISTORY_ROWS,
    'history_load_status': history_load_status,
    'history_load_reason': history_load_reason or None,
    'method': chosen_method,
    'llm_provider_model': llm_provider_used,
    'metrics': {
        'baseline': baseline_metrics,
        'llm': llm_metrics,
    },
    'predicted_range': predicted_range,
    'confidence_level': confidence_level,
    'limitations': limitations,
    'generated_at': datetime.now(timezone.utc).isoformat(),
}

viral_prediction_path = RESULTS_DIR / 'viral_prediction.json'
with open(viral_prediction_path, 'w', encoding='utf-8') as f:
    json.dump(viral_prediction, f, ensure_ascii=False, indent=2)
print('Kaydedildi:', viral_prediction_path, '(status:', prediction_status, ')')

zip_path = RESULTS_DIR / '05_viral_prediction_paylasim_paketi.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(viral_prediction_path, arcname='viral_prediction.json')
    if out_path.exists():
        archive.write(out_path, arcname=out_path.name)
print('Paylasim paketi hazir:', zip_path)

## 8. Karar (elle doldurulacak)

Not: Gerçek üretim verisi biriktikçe (Faz 9 — Öğrenme Döngüsü) bu karar `ai_call_logs` + `analytics_snapshots` verisiyle yeniden değerlendirilecek. `MIN_HISTORY_ROWS` eşiği altında hiçbir yöntem `production_ready` sayılmamalıdır.

- **Seçilen yaklaşım:** _...
- **Gerekçe:** _...
